# Tip–Tap: persistent crop stress and agricultural land-use change in the Kashgar oasis, 2019–2025

# Part 1 · Core analysis

## 1.1 · Configuration and frozen parameters

In [ ]:
from __future__ import annotations
import os, re, json, glob, hashlib, warnings, platform, sys, time
from pathlib import Path
from dataclasses import dataclass, asdict, field

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape, Point
import rasterio
from rasterio import features as rfeatures
from scipy.spatial import cKDTree
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 80)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

DEV_SAMPLE       = os.environ.get("TIPTAP_DEV", "0") == "1"      # True: one *_000 chunk per year (quick check). Set False for the full panel.
RUN_GEE_EXPORTS  = os.environ.get("TIPTAP_GEE", "0") == "1"      # §2 only
DATA_ROOT        = Path(os.environ.get("TIPTAP_DATA_ROOT", r"D:\BRA"))
OUT              = Path(os.environ.get("TIPTAP_OUT", str(DATA_ROOT / "TipTap" / "outputs_recovery")))
# every savefig(...png) also writes a vector PDF next to it (journal line art); Fig. 1a (127k points) is best used as PNG
_orig_savefig = plt.Figure.savefig
def _dual_savefig(self, fname, *a, **k):
    _orig_savefig(self, fname, *a, **k)
    if str(fname).endswith(".png"): _orig_savefig(self, str(fname)[:-4] + ".pdf", *a, **k)
plt.Figure.savefig = _dual_savefig
for sub in ["interim", "tables", "figures", "gis"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

ZONAL_DIR = next(iter(DATA_ROOT.glob("TipTap_Kashgar*/TipTap_Kashgar")), DATA_ROOT / "TipTap_Kashgar")
PATHS = dict(
    zonal_dir   = ZONAL_DIR,
    cell_lookup = ZONAL_DIR / "tiptap_cell_lookup.csv",
    crop_dir    = DATA_ROOT / "Data" / "Kashgar_crops_HLS30",
    pmf_2020    = DATA_ROOT / "Data" / "Public" / "China-PMF-10_2020.tif",
    osm         = DATA_ROOT / "Data" / "Gee_Data" / "social_financial_capital" / "osm" / "osm_all_features.gpkg",
    adm3        = DATA_ROOT / "Data" / "chn_adm_ocha_2020_shp" / "geoBoundaries-CHN-ADM3_simplified.shp",
    prefecture  = DATA_ROOT / "Data" / "kashgar_perfecture.shp",
    viirs_dir   = DATA_ROOT / "Data" / "Gee_Data" / "financial_capital" / "viirs_ntl",
    era5_block  = OUT / "interim" / "era5_cell_12day.csv",     # produced by the optional GEE cell in §2
)

PARAMS = dict(
    seed                = 20260913,
    years               = list(range(2019, 2026)),
    baseline_years      = [2019, 2020, 2021, 2022],          # climatology + training window
    test_years          = [2023, 2024, 2025],                # forward validation + targeting evaluation
    composite_days      = 12,
    season_cidx         = (5, 19),        # May–Oct composites used for episode detection (index on the Mar–Nov grid)
    min_pure_px_10      = 6,              # R6 rule: ≥ 6 interior 10 m pixels
    z_stress_evi2       = -1.0,           # [tune] vigour drop
    z_corrob_water      = -0.5,           # [tune] NDMI or ΔVV corroboration
    z_recovery          = -0.5,           # [tune] two consecutive composites at or above -> recovered
    min_persist         = 2,              # composites (24 d) to confirm onset
    pre_onset_clear     = 2,              # composites of S=0 required before onset for an 'acute' episode
    fallow_evi2_max     = 0.25,           # [tune] season max EVI2 below this -> not cultivated
    recovery_horizons_d = [24, 36, 48],   # horizons at which P_rec is reported
    rci_horizon_d       = 36,             # horizon defining the Recovery Capacity Index
    size_bins_ha        = [0, 0.5, 2, 10, np.inf],
    size_labels         = ["<0.5 ha", "0.5–2 ha", "2–10 ha", ">10 ha"],
    hydraulic_max_snap_m= 3000,           # parcels farther than this from any mapped canal are flagged 'unmapped'
    crop_legend = {1: "spring maize", 2: "summer maize", 3: "rice", 4: "winter wheat", 5: "spring wheat",
                   6: "cotton", 7: "other economic", 8: "orchard", 0: "unclassified", -128: "nodata"},
    # Nominal Kashgar crop calendars (DOY): initial | development | mid-season | late-season | harvest.
    # VERIFY against local extension records - they define 'onset stage'.
    crop_calendar = {"cotton": (105, 145, 190, 250, 290), "winter wheat": (280, 75, 120, 160, 185),
                     "spring maize": (110, 150, 195, 235, 265), "summer maize": (170, 200, 235, 265, 290),
                     "orchard": (90, 120, 160, 260, 300), "other economic": (110, 150, 195, 250, 290),
                     "rice": (120, 160, 205, 245, 275), "spring wheat": (80, 115, 150, 185, 205)},
)
np.random.seed(PARAMS["seed"])
GRID_START_DOY = 60          # 1 March on the export grid
def cidx_to_doy(c): return GRID_START_DOY + PARAMS["composite_days"] * np.asarray(c)

MANIFEST = dict(run_started=time.strftime("%Y-%m-%dT%H:%M:%S"), dev_sample=DEV_SAMPLE, python=sys.version.split()[0],
                platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__, geopandas=gpd.__version__,
                params_hash=hashlib.md5(json.dumps(PARAMS, sort_keys=True, default=str).encode()).hexdigest()[:10],
                params=PARAMS, data_root=str(DATA_ROOT))
print(f"data root : {DATA_ROOT}\noutputs   : {OUT}\nzonal dir : {ZONAL_DIR}\nDEV_SAMPLE={DEV_SAMPLE}  RUN_GEE_EXPORTS={RUN_GEE_EXPORTS}  params_hash={MANIFEST['params_hash']}")


## 1.2 · Parcel panel: ingest, geometry and mulch index

In [ ]:
VARS = ["EVI2", "MSAVI2", "NDMI", "NDRE", "MNDWI", "MNDWImax", "SI", "VV", "VH", "CR", "VVrange", "S1N", "S2N"]

def zonal_files(year):
    pat = f"tiptap_zonal_{year}_000.csv" if DEV_SAMPLE else f"tiptap_zonal_{year}_*.csv"
    return sorted(PATHS["zonal_dir"].glob(pat))

def wide_to_long(df: pd.DataFrame, year: int) -> pd.DataFrame:
    tv = [c for c in df.columns if re.search(r"_\d{8}_(mean|sd)$", c)]
    m = df[["parcel_id"] + tv].melt(id_vars="parcel_id", var_name="k", value_name="v").dropna(subset=["v"])
    ex = m["k"].str.extract(r"^(?P<var>[A-Za-z0-9]+)_(?P<date>\d{8})_(?P<stat>mean|sd)$")
    m = pd.concat([m.drop(columns="k"), ex], axis=1)
    m["col"] = m["var"] + "_" + m["stat"]
    w = m.pivot_table(index=["parcel_id", "date"], columns="col", values="v", aggfunc="first").reset_index()
    w["date"] = pd.to_datetime(w["date"]); w["year"] = year
    w["cidx"] = ((w["date"] - pd.Timestamp(f"{year}-03-01")).dt.days // PARAMS["composite_days"]).astype(int)
    return w

def ingest_year(year: int, force=False) -> pd.DataFrame:
    tag = "dev" if DEV_SAMPLE else "full"
    pq = OUT / "interim" / f"panel_{tag}_{year}.parquet"
    if pq.exists() and not force:
        return pd.read_parquet(pq)
    parts, metas = [], []
    for f in zonal_files(year):
        df = pd.read_csv(f)
        parts.append(wide_to_long(df, year))
        metas.append(df[["parcel_id", "area_ha", "pure_px_10", ".geo"]])
        print(f"  {f.name}: {len(df)} parcels", end="\r")
    long = pd.concat(parts, ignore_index=True)
    long.to_parquet(pq, index=False)
    meta = pd.concat(metas, ignore_index=True).drop_duplicates("parcel_id")
    meta.to_parquet(OUT / "interim" / f"parcel_meta_{tag}_{year}.parquet", index=False)
    return long

t0 = time.time()
PANEL = pd.concat([ingest_year(y) for y in PARAMS["years"]], ignore_index=True)
PANEL = PANEL.sort_values(["parcel_id", "year", "cidx"]).reset_index(drop=True)
print(f"\npanel: {len(PANEL):,} parcel-composites, {PANEL.parcel_id.nunique():,} parcels, years {PANEL.year.min()}–{PANEL.year.max()}  ({time.time()-t0:.0f}s)")

# --- parcel geometry & static metadata (from the export itself) --------------------------------
tag = "dev" if DEV_SAMPLE else "full"
meta = pd.concat([pd.read_parquet(p) for p in sorted((OUT / "interim").glob(f"parcel_meta_{tag}_*.parquet"))]).drop_duplicates("parcel_id")
PARCELS = gpd.GeoDataFrame(meta.drop(columns=".geo"), geometry=[shape(json.loads(s)) for s in meta[".geo"]], crs="EPSG:4326")
PARCELS["geometry"] = PARCELS.geometry.buffer(0)               # repair GeometryCollections / self-touching rings
PARCELS = PARCELS[PARCELS.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
PARCELS = PARCELS[PARCELS.pure_px_10 >= PARAMS["min_pure_px_10"]].reset_index(drop=True)
PARCELS["size_class"] = pd.cut(PARCELS.area_ha, PARAMS["size_bins_ha"], labels=PARAMS["size_labels"], right=False)
PARCELS["log_area"] = np.log(PARCELS.area_ha)
PANEL = PANEL[PANEL.parcel_id.isin(PARCELS.parcel_id)]

# --- ERA5 cell membership (9 km block) ----------------------------------------------------------
lk = pd.read_csv(PATHS["cell_lookup"], usecols=["cell", "parcel_id"]).drop_duplicates("parcel_id")
PARCELS = PARCELS.merge(lk, on="parcel_id", how="left")
PARCELS["cell"] = PARCELS["cell"].fillna("unknown")

# --- plastic-mulch index per parcel-year --------------------------------------------------------
pm = []
for y in PARAMS["years"]:
    f = PATHS["zonal_dir"] / f"tiptap_pmli_{y}.csv"
    if f.exists():
        d = pd.read_csv(f, usecols=["parcel_id", "PMLI", "NDVI_apr"]); d["year"] = y; pm.append(d)
PMLI = pd.concat(pm, ignore_index=True) if pm else pd.DataFrame(columns=["parcel_id", "PMLI", "NDVI_apr", "year"])
# keep only the columns used downstream (the full panel is 19 M rows; this cuts memory by ~3x)
PANEL = PANEL[["parcel_id", "year", "cidx", "date", "EVI2_mean", "NDMI_mean", "VV_mean", "SI_mean", "MNDWImax_mean", "S2N_mean", "S1N_mean"]]
print(f"parcels retained: {len(PARCELS):,} | median area {PARCELS.area_ha.median():.2f} ha | size classes: {PARCELS.size_class.value_counts().to_dict()}")
print(f"PMLI years on disk: {sorted(PMLI.year.unique().tolist())}")


## 1.3 · Within-parcel EVI2 heterogeneity and seasonal maximum

Per parcel-year peak-season EVI2 mean and SD and the seasonal maximum (cultivation rule); read by Parts 2 and 3.

In [ ]:
#within-parcel EVI2 heterogeneity and seasonal maximum per parcel-year
import pyarrow.parquet as pq

REV_DIR = OUT / "revision"; REV_DIR.mkdir(parents=True, exist_ok=True)
c0, c1 = PARAMS["season_cidx"]                     # May–Oct composites, as used for episode detection
parts = []
for y in PARAMS["years"]:
    f = OUT / "interim" / f"panel_full_{y}.parquet"
    if not f.exists():
        print(f"{f.name}: missing — skipped"); continue
    cols = pq.read_schema(f).names
    need = ["parcel_id", "year", "cidx", "EVI2_mean", "EVI2_sd"]
    missing = [c for c in need if c not in cols]
    assert not missing, f"{f.name} lacks {missing}; columns present: {cols[:20]} ..."
    d = pq.read_table(f, columns=need).to_pandas().dropna(subset=["EVI2_mean"])
    emax = d.groupby(["parcel_id", "year"]).EVI2_mean.max().rename("evi2_max_all").reset_index()
    d = d[(d.cidx >= c0) & (d.cidx <= c1)]
    idx = d.groupby(["parcel_id", "year"]).EVI2_mean.idxmax()
    pk = (d.loc[idx, ["parcel_id", "year", "cidx", "EVI2_mean", "EVI2_sd"]]
            .rename(columns={"cidx": "peak_cidx", "EVI2_mean": "evi2_peak_mean", "EVI2_sd": "evi2_peak_sd"}))
    nobs = d.groupby(["parcel_id", "year"]).EVI2_mean.size().rename("n_obs").reset_index()
    pk = pk.merge(nobs, on=["parcel_id", "year"]).merge(emax, on=["parcel_id", "year"], how="left")
    parts.append(pk); print(f"{y}: {len(pk):,} parcel-years")
    del d
HET = pd.concat(parts, ignore_index=True)
HET.to_csv(REV_DIR / "R11_parcel_heterogeneity.csv", index=False)
print(f"written {REV_DIR / 'R11_parcel_heterogeneity.csv'}  ({len(HET):,} rows)")
print("cultivated share at EVI2max ≥ 0.20 / 0.25 / 0.30:",
      {t: round(float((HET.evi2_max_all >= t).mean()), 3) for t in (0.20, 0.25, 0.30)})

## 1.4 · Structural covariates from public layers

In [ ]:
LOCAL_CRS = "+proj=aea +lat_1=35 +lat_2=45 +lat_0=40 +lon_0=77 +datum=WGS84 +units=m +no_defs"
P_loc = PARCELS.to_crs(LOCAL_CRS)
CENT = P_loc.copy(); CENT["geometry"] = P_loc.geometry.centroid

# 4.1 County ---------------------------------------------------------------------------------------
adm3 = gpd.read_file(PATHS["adm3"]).to_crs(LOCAL_CRS)[["shapeName", "geometry"]].rename(columns={"shapeName": "county"})
PARCELS["county"] = gpd.sjoin(CENT[["parcel_id", "geometry"]], adm3, how="left", predicate="within").drop_duplicates("parcel_id").set_index("parcel_id").loc[PARCELS.parcel_id, "county"].fillna("unassigned").values

# 4.2 Crop class per year (majority of 30 m pixels inside the parcel) -----------------------------
def raster_majority(parcels_wgs: gpd.GeoDataFrame, tif: Path, valuename="cls", tile_deg=0.5, binary_mean=False):
    """Rasterise parcel ids onto the raster grid tile by tile; return per-parcel majority class + purity
    (or mean value when binary_mean=True)."""
    out = []
    cx = (parcels_wgs.geometry.centroid.x // tile_deg).astype(int); cy = (parcels_wgs.geometry.centroid.y // tile_deg).astype(int)
    with rasterio.open(tif) as r:
        for (tx, ty), idx in parcels_wgs.groupby([cx, cy]).groups.items():
            sub = parcels_wgs.loc[idx]
            b = sub.total_bounds
            win = rasterio.windows.from_bounds(*b, transform=r.transform).round_offsets().round_lengths()
            win = win.intersection(rasterio.windows.Window(0, 0, r.width, r.height))
            if win.width <= 0 or win.height <= 0: continue
            arr = r.read(1, window=win); tr = r.window_transform(win)
            ids = rfeatures.rasterize(((g, i) for g, i in zip(sub.geometry, range(1, len(sub) + 1))),
                                      out_shape=arr.shape, transform=tr, fill=0, dtype="int32")
            m = ids > 0
            if not m.any(): continue
            df = pd.DataFrame({"i": ids[m], "v": arr[m].astype(float)})
            if binary_mean:
                g = df.groupby("i").v.agg(["mean", "size"]); g.columns = [valuename, "npx"]
            else:
                ct = pd.crosstab(df.i, df.v)
                g = pd.DataFrame({valuename: ct.idxmax(axis=1).astype(int), "purity": ct.max(axis=1) / ct.sum(axis=1), "npx": ct.sum(axis=1)})
            g["parcel_id"] = sub.parcel_id.values[g.index.values - 1]
            out.append(g.reset_index(drop=True))
    return pd.concat(out, ignore_index=True)

crop_years = sorted(int(re.search(r"_(\d{4})_", f.name).group(1)) for f in PATHS["crop_dir"].glob("Kashgar_crops_*_HLS30_WGS84.tif"))
CROP = []
for y in PARAMS["years"]:
    ysrc = max([cy for cy in crop_years if cy <= y] or [min(crop_years)])          # 2025 -> 2024 map
    c = raster_majority(PARCELS, PATHS["crop_dir"] / f"Kashgar_crops_{ysrc}_HLS30_WGS84.tif", "crop_cls")
    c["year"] = y; c["crop_map_year"] = ysrc; CROP.append(c)
CROP = pd.concat(CROP, ignore_index=True)
CROP["crop"] = CROP.crop_cls.map(PARAMS["crop_legend"]).fillna("unclassified")
CROP.loc[CROP.crop.isin(["nodata"]), "crop"] = "unclassified"
print("crop classes (parcel-years):", CROP.crop.value_counts().to_dict())

# 4.3 Mulch flag: calibrate PMLI sign and threshold against the independent 2020 mulch map ---------
pmf = raster_majority(PARCELS, PATHS["pmf_2020"], "pmf_frac", binary_mean=True)[["parcel_id", "pmf_frac"]]
cal_year = 2020 if 2020 in PMLI.year.unique() else int(PMLI.year.min())
cal = PMLI[PMLI.year == cal_year].merge(pmf, on="parcel_id").dropna()
sign = np.sign(np.corrcoef(cal.PMLI, cal.pmf_frac)[0, 1]) or 1.0
grid = np.quantile(sign * cal.PMLI, np.linspace(0.05, 0.95, 91))
truth = cal.pmf_frac > 0.5
def bal_acc(thr):
    pred = sign * cal.PMLI > thr
    tpr = (pred & truth).sum() / max(truth.sum(), 1); tnr = (~pred & ~truth).sum() / max((~truth).sum(), 1)
    return 0.5 * (tpr + tnr)
ba = np.array([bal_acc(t) for t in grid]); PMLI_THR = float(grid[ba.argmax()]); PMLI_SIGN = float(sign)
MANIFEST["mulch_calibration"] = dict(year=cal_year, sign=PMLI_SIGN, threshold=PMLI_THR, balanced_accuracy=float(ba.max()),
                                     corr_pmli_pmf=float(np.corrcoef(cal.PMLI, cal.pmf_frac)[0, 1]), n=int(len(cal)))
PMLI["mulch"] = (PMLI_SIGN * PMLI.PMLI > PMLI_THR).astype(int)
print(f"PMLI calibration vs PMF-2020: sign={PMLI_SIGN:+.0f}, threshold={PMLI_THR:.3f}, balanced accuracy={ba.max():.2f}, r={MANIFEST['mulch_calibration']['corr_pmli_pmf']:.2f} (n={len(cal)})")

# 4.4 Hydraulic remoteness on the OSM water network ------------------------------------------------
osm = gpd.read_file(PATHS["osm"], columns=["fclass", "name"]).to_crs(LOCAL_CRS)
lines = osm[osm.fclass.isin(["river", "canal", "drain", "stream"]) & osm.geometry.geom_type.isin(["LineString", "MultiLineString"])].explode(index_parts=False)
G = nx.Graph(); river_nodes, canal_nodes = set(), []
for fclass, geom in zip(lines.fclass, lines.geometry):
    xy = np.round(np.asarray(geom.coords)[:, :2], 0)
    nodes = [tuple(p) for p in xy]
    for a, b in zip(nodes[:-1], nodes[1:]):
        G.add_edge(a, b, w=float(np.hypot(a[0] - b[0], a[1] - b[1])))
    if fclass == "river": river_nodes.update(nodes)
    else: canal_nodes.extend(nodes)                   # canals, drains and streams = distribution network
net_d = nx.multi_source_dijkstra_path_length(G, [n for n in river_nodes if n in G], weight="w") if river_nodes else {}
canal_nodes = np.array(sorted(set(canal_nodes)))
canal_netd = np.array([net_d.get(tuple(n), np.nan) for n in canal_nodes])
tree = cKDTree(canal_nodes)
cxy = np.c_[CENT.geometry.x, CENT.geometry.y]
snap_d, snap_i = tree.query(cxy, k=1)
river_xy = np.array(sorted(river_nodes)); rtree = cKDTree(river_xy)
river_d, _ = rtree.query(cxy, k=1)
PARCELS["dist_canal_m"] = snap_d
PARCELS["dist_river_m"] = river_d
netd = canal_netd[snap_i]
PARCELS["hydraulic_net_m"] = np.where(np.isfinite(netd), netd + snap_d, np.nan)
PARCELS["hyd_unmapped"] = ((snap_d > PARAMS["hydraulic_max_snap_m"]) | ~np.isfinite(netd)).astype(int)
# Remoteness index used in models: log network distance for connected parcels; fallback log straight-line river distance
PARCELS["hyd_remote_km"] = np.where(PARCELS.hyd_unmapped == 0, PARCELS.hydraulic_net_m, PARCELS.dist_river_m) / 1000
PARCELS["log_hyd_remote"] = np.log1p(PARCELS.hyd_remote_km)
PARCELS["hyd_tercile"] = PARCELS.groupby("county")["hyd_remote_km"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["head", "middle", "tail"]) if s.notna().sum() >= 30 else pd.Series(["middle"] * len(s), index=s.index))
print(f"OSM water graph: {G.number_of_nodes():,} nodes, {len(river_nodes):,} river source nodes, {len(canal_nodes):,} distribution-network nodes | parcels unmapped (> {PARAMS['hydraulic_max_snap_m']} m from any mapped channel): {PARCELS.hyd_unmapped.mean():.1%}")
MANIFEST["hydraulic_unmapped_share"] = float(PARCELS.hyd_unmapped.mean())

# 4.4b Oasis-fringe position: distance from the parcel to the desert margin (oasis mask = cropland fraction >= 0.15 at ~480 m, closed and opened, from the 30 m crop map)
from scipy import ndimage
def fringe_distance(parcels_wgs, tif, agg=16, min_frac=0.15):
    with rasterio.open(tif) as r:
        arr = r.read(1); tr = r.transform
    crop = np.isin(arr, list(range(1, 9))).astype(np.uint8)
    h, w = (crop.shape[0] // agg) * agg, (crop.shape[1] // agg) * agg
    frac = crop[:h, :w].reshape(h // agg, agg, w // agg, agg).mean(axis=(1, 3))
    mask = ndimage.binary_opening(ndimage.binary_closing(frac >= min_frac, iterations=2), iterations=1)   # ~480 m oasis mask
    px_m = abs(tr.a) * agg * 111_000 * np.cos(np.deg2rad(39.5))          # metres per coarse pixel (E-W)
    dist = ndimage.distance_transform_edt(mask) * px_m                    # distance to nearest non-cropland
    cx = ((parcels_wgs.geometry.centroid.x - tr.c) / (tr.a * agg)).astype(int).clip(0, mask.shape[1] - 1)
    cy = ((parcels_wgs.geometry.centroid.y - tr.f) / (tr.e * agg)).astype(int).clip(0, mask.shape[0] - 1)
    return dist[cy.values, cx.values] / 1000
fringe_year = min(crop_years, key=lambda y: abs(y - 2021))
PARCELS["dist_desert_km"] = fringe_distance(PARCELS, PATHS["crop_dir"] / f"Kashgar_crops_{fringe_year}_HLS30_WGS84.tif")
PARCELS["log_dist_desert"] = np.log1p(PARCELS.dist_desert_km)
PARCELS["fringe_tercile"] = PARCELS.groupby("county")["dist_desert_km"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["fringe", "middle", "core"]) if s.notna().sum() >= 30 else pd.Series(["middle"] * len(s), index=s.index))

# 4.5 Settlement, road access and night-time light ----------------------------------------------
places = osm[osm.fclass.isin(["city", "town", "village", "hamlet"]) & (osm.geometry.geom_type == "Point")]
PARCELS["dist_settlement_km"] = cKDTree(np.c_[places.geometry.x, places.geometry.y]).query(cxy)[0] / 1000
roads = osm[osm.fclass.isin(["motorway", "trunk", "primary", "secondary"]) & osm.geometry.geom_type.isin(["LineString", "MultiLineString"])].explode(index_parts=False)
rxy = np.vstack([np.asarray(g.coords)[:, :2] for g in roads.geometry]) if len(roads) else np.zeros((1, 2))
PARCELS["dist_road_km"] = cKDTree(rxy).query(cxy)[0] / 1000
def sample_raster(tif, pts_wgs):
    with rasterio.open(tif) as r:
        return np.array([v[0] for v in r.sample([(p.x, p.y) for p in pts_wgs])], dtype=float)
cent_wgs = CENT.to_crs("EPSG:4326").geometry
ntl_files = {int(re.search(r"(\d{4})", f.name).group(1)): f for f in PATHS["viirs_dir"].glob("viirs_ntl_*.tif")}
ntl_y = min(ntl_files, key=lambda y: abs(y - 2021)) if ntl_files else None
PARCELS["ntl"] = sample_raster(ntl_files[ntl_y], cent_wgs) if ntl_y else np.nan
PARCELS["log_ntl"] = np.log1p(PARCELS.ntl.clip(lower=0))

# 4.6 Pre-season salinity class (bare-soil SI, March composites, median over years) ----------------
si = PANEL[PANEL.cidx <= 2].groupby("parcel_id").SI_mean.median().rename("si_preseason")
PARCELS = PARCELS.merge(si, on="parcel_id", how="left")
PARCELS["salinity_class"] = PARCELS.groupby("county")["si_preseason"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["low", "mid", "high"]) if s.notna().sum() >= 30 else pd.Series(["mid"] * len(s), index=s.index))

PARCELS.to_parquet(OUT / "interim" / f"parcels_struct_{tag}.parquet")
display(PARCELS[["area_ha", "hyd_remote_km", "dist_desert_km", "dist_settlement_km", "dist_road_km", "ntl", "si_preseason"]].describe().round(2))


## 1.5 · Directed hydraulic position (ArcGIS Pro)

In [ ]:
HYD_CSV = DATA_ROOT / "TipTap" / "hydraulic" / "hydraulic_position.csv"
HAS_HYD = HYD_CSV.exists()
HYD_TERMS = "hyd_rank + hyd_dz_m" if HAS_HYD else "hyd_rank"        # model terms used in §8–§10
HYD_NUM   = ["hyd_rank", "hyd_dz_m"] if HAS_HYD else ["hyd_rank"]
if HAS_HYD:
    hyd = pd.read_csv(HYD_CSV)
    drop = [c for c in hyd.columns if c != "parcel_id" and c in PARCELS.columns]
    PARCELS = PARCELS.drop(columns=drop).merge(hyd, on="parcel_id", how="left")
    PARCELS["hyd_tier"] = PARCELS.hyd_tier.fillna(3).astype(int)
    PARCELS["hyd_mapped"] = (PARCELS.hyd_tier == 1).astype(int)
    PARCELS["hyd_unmapped"] = (PARCELS.hyd_tier != 1).astype(int)
    # main variable: tier-1 head-to-tail rank (within the canal command where >= 5 parcels share an intake, else within county)
    PARCELS["hyd_rank"] = PARCELS.hyd_rank_cmd.fillna(PARCELS.hyd_rank_county)
    # sensitivity variable: tier-1 rank extended with the tier-2 (undirected) rank where tier 1 is unavailable
    PARCELS["hyd_rank_t12"] = PARCELS.hyd_rank.fillna(PARCELS.hyd_rank_cmd_t2).fillna(PARCELS.hyd_rank_county_t2)
    for c in ["hyd_rank", "hyd_rank_t12"]:          # unreached parcels: county median; their flag carries the information
        PARCELS[c] = PARCELS[c].fillna(PARCELS.groupby("county")[c].transform("median")).fillna(0.5)
    PARCELS["hyd_dz_m"] = PARCELS.hyd_dz_m.fillna(PARCELS.groupby("county").hyd_dz_m.transform("median")).fillna(0)
    PARCELS["hyd_remote_km"] = np.where(PARCELS.hyd_tier == 1, PARCELS.hyd_net_dir_m / 1000,
                               np.where(PARCELS.hyd_tier == 2, PARCELS.hyd_net_undir_m / 1000, PARCELS.hyd_remote_km))
    PARCELS["hyd_tercile"] = PARCELS.groupby("county")["hyd_rank"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["head", "middle", "tail"]))
    MANIFEST["hydraulic_source"] = "ArcGIS Pro-prepared directed water network (Copernicus GLO-30 edge directions, Dijkstra from river and component-head intakes)"
    MANIFEST["hydraulic_tiers"] = {int(k): float(v) for k, v in PARCELS.hyd_tier.value_counts(normalize=True).round(3).items()}
    MANIFEST["hydraulic_unmapped_share"] = float(PARCELS.hyd_unmapped.mean())
    print("hydraulic tiers:", MANIFEST["hydraulic_tiers"], "| hyd_rank median", round(PARCELS.hyd_rank.median(), 2), "| hyd_dz_m median", round(PARCELS.hyd_dz_m.median(), 1), "m")
else:
    # fallback: head-to-tail rank from the undirected §4.4 distance, within county; no DEM fallback available
    PARCELS["hyd_rank"] = PARCELS.groupby("county")["hyd_remote_km"].rank(pct=True).fillna(0.5)
    PARCELS["hyd_rank_t12"] = PARCELS["hyd_rank"]; PARCELS["hyd_dz_m"] = 0.0
    MANIFEST["hydraulic_source"] = "undirected OSM proxy (§4.4) ranked within county — hydraulic_position.csv not found"
    print("hydraulic_position.csv not found — hyd_rank = within-county rank of the undirected §4.4 distance; hyd_dz_m unavailable")


## 1.6 · Phenology-normalised anomalies

In [ ]:
D = PANEL.merge(CROP[["parcel_id", "year", "crop", "purity"]], on=["parcel_id", "year"], how="left")
D["crop"] = D["crop"].fillna("unclassified")
D = D.merge(PARCELS[["parcel_id", "county", "cell"]], on="parcel_id", how="left")
D = D.sort_values(["parcel_id", "year", "cidx"]).reset_index(drop=True)

# single-composite gaps in the optical record are linearly interpolated (limit=1); longer gaps stay missing
for v in ["EVI2_mean", "NDMI_mean"]:
    D[v] = D.groupby(["parcel_id", "year"])[v].transform(lambda s: s.interpolate(limit=1, limit_area="inside"))
D["dVV"] = D.groupby(["parcel_id", "year"])["VV_mean"].diff()

def robust_z(df, col, keys, base_mask, min_n=30):
    b = df.loc[base_mask, keys + [col]].dropna()
    g = b.groupby(keys)[col]
    stats = pd.DataFrame({"med": g.median(), "mad": g.apply(lambda s: (s - s.median()).abs().median()) * 1.4826, "n": g.size()}).reset_index()
    stats.loc[stats.n < min_n, ["med", "mad"]] = np.nan
    fb = b.groupby(["crop", "cidx"])[col]
    fallback = pd.DataFrame({"med_fb": fb.median(), "mad_fb": fb.apply(lambda s: (s - s.median()).abs().median()) * 1.4826}).reset_index()
    out = df[keys].merge(stats, on=keys, how="left").merge(fallback, on=["crop", "cidx"], how="left")
    med = out.med.fillna(out.med_fb).values; mad = out.mad.fillna(out.mad_fb).replace(0, np.nan).values
    return (df[col].values - med) / mad

base = D.year.isin(PARAMS["baseline_years"])
D["z_evi2"] = robust_z(D, "EVI2_mean", ["crop", "county", "cidx"], base)
D["z_ndmi"] = robust_z(D, "NDMI_mean", ["crop", "county", "cidx"], base)
D["z_dvv"]  = robust_z(D, "dVV",       ["crop", "county", "cidx"], base)

# parcel-year summaries: cultivation status, season integral
PY = (D.groupby(["parcel_id", "year"]).agg(evi2_max=("EVI2_mean", "max"), evi2_int=("EVI2_mean", "sum"), n_obs=("EVI2_mean", "count"),
                                           crop=("crop", "first"), county=("county", "first"), cell=("cell", "first")).reset_index())
PY["cultivated"] = (PY.evi2_max >= PARAMS["fallow_evi2_max"]).astype(int)
PY["z_int"] = PY.groupby(["crop", "county"]).evi2_int.transform(lambda s: (s - s.median()) / (1.4826 * (s - s.median()).abs().median() + 1e-9))
print(f"anomalies computed for {len(D):,} rows; cultivated parcel-years: {PY.cultivated.mean():.1%}; "
      f"z_evi2 quantiles (baseline): {np.nanpercentile(D.loc[base,'z_evi2'], [5,25,50,75,95]).round(2)}")


## 1.7 · Majority crop class per parcel and crop-map year (2019–2024)

In [ ]:
# R12/R13 helper: majority crop class and purity per parcel per crop-map year (2019–2024)
CM_OUT = OUT / "revision" / "R12_R13"; CM_OUT.mkdir(parents=True, exist_ok=True)
_m = pd.read_parquet(OUT / "interim" / "parcel_meta_full_2023.parquet")
CM_P = gpd.GeoDataFrame(_m[["parcel_id"]], geometry=[shape(json.loads(s)) for s in _m[".geo"]], crs="EPSG:4326")
CM_P["geometry"] = CM_P.geometry.buffer(0)
CM_LEG = {1: "spring maize", 2: "summer maize", 3: "rice", 4: "winter wheat", 5: "spring wheat", 6: "cotton", 7: "other economic", 8: "orchard", 0: "unclassified", -128: "nodata"}
_cm = []
for y in range(2019, 2025):
    c = raster_majority(CM_P, PATHS["crop_dir"] / f"Kashgar_crops_{y}_HLS30_WGS84.tif", "crop_cls"); c["map_year"] = y
    _cm.append(c); print(y, len(c), flush=True)
CM = pd.concat(_cm, ignore_index=True)
CM["crop"] = CM.crop_cls.map(CM_LEG).fillna("unclassified"); CM.loc[CM.crop == "nodata", "crop"] = "unclassified"
CM.to_parquet(CM_OUT / "crop_majority_2019_2024.parquet", index=False)
print(CM.groupby("map_year").crop.value_counts().unstack().fillna(0).astype(int))
del _m, _cm, CM_P

## 1.8 · Anomaly-baseline strata (crop × county × composite, 2019–2022)

In [ ]:
# --- R20 (full panel, self-contained): anomaly-baseline strata crop x county x composite, 2019-2022 ----------
import pyarrow.parquet as pq
REV_DIR = OUT / "revision"; REV_DIR.mkdir(parents=True, exist_ok=True)
S_ = pd.read_parquet(OUT / "interim" / "parcels_struct_full.parquet", columns=["parcel_id", "county"])
CR_ = pd.read_parquet(REV_DIR / "R12_R13" / "crop_majority_2019_2024.parquet", columns=["parcel_id", "map_year", "crop"]).rename(columns={"map_year": "year"})
parts = []
for y in PARAMS["baseline_years"]:
    d = pq.read_table(OUT / "interim" / f"panel_full_{y}.parquet", columns=["parcel_id", "year", "cidx", "EVI2_mean", "NDMI_mean", "VV_mean"]).to_pandas()
    d = d.sort_values(["parcel_id", "cidx"])
    for v in ["EVI2_mean", "NDMI_mean"]:
        d[v] = d.groupby("parcel_id")[v].transform(lambda s: s.interpolate(limit=1, limit_area="inside"))
    d["dVV"] = d.groupby("parcel_id")["VV_mean"].diff()
    parts.append(d.drop(columns="VV_mean")); print(y, len(d), flush=True)
B = pd.concat(parts, ignore_index=True).merge(CR_, on=["parcel_id", "year"], how="left").merge(S_, on="parcel_id", how="left")
B["crop"] = B.crop.fillna("unclassified")
rows = []
for col in ["EVI2_mean", "NDMI_mean", "dVV"]:
    b = B[["crop", "county", "cidx", col]].dropna()
    g = b.groupby(["crop", "county", "cidx"])[col]
    st = pd.DataFrame({"n": g.size(), "median": g.median(),
                       "mad": g.apply(lambda s: (s - s.median()).abs().median()) * 1.4826}).reset_index()
    st["variable"] = col; st["fallback_n_lt_30"] = st.n < 30; st["mad_zero"] = st.mad == 0
    rows.append(st)
STRATA = pd.concat(rows, ignore_index=True); STRATA.to_csv(REV_DIR / "R20_baseline_strata.csv", index=False)
summ = STRATA.groupby("variable").agg(strata=("n", "size"), median_n=("n", "median"), min_n=("n", "min"),
        strata_fallback=("fallback_n_lt_30", "sum"), strata_mad_zero=("mad_zero", "sum"), obs_total=("n", "sum"))
summ["obs_in_fallback"] = STRATA[STRATA.fallback_n_lt_30].groupby("variable").n.sum()
summ["share_obs_fallback"] = summ.obs_in_fallback / summ.obs_total
print(summ.round(4).to_string())
print(STRATA[STRATA.variable == "EVI2_mean"].groupby("crop").n.median().round(0).to_dict())
del B, parts

## 1.9 · Tip and Tap: stress states, episodes, episode types and block-level forcing

In [ ]:
# --- growing window per crop on the composite grid --------------------------------------------------
def crop_window(crop):
    cal = PARAMS["crop_calendar"].get(crop)
    if cal is None:                                   # unclassified: use the generic May–Oct window
        return PARAMS["season_cidx"]
    plant, dev, mid, late, harvest = cal
    start = 0 if plant > harvest else max(0, int((dev - GRID_START_DOY) // PARAMS["composite_days"]) - 1)   # winter crops start at grid start
    end = min(22, int((harvest - GRID_START_DOY) // PARAMS["composite_days"]))
    return (max(start, 0), max(end, start + 3))
def onset_stage(crop, doy):
    cal = PARAMS["crop_calendar"].get(crop)
    if cal is None: return "unknown"
    plant, dev, mid, late, harvest = cal
    if doy < dev: return "initial"
    if doy < mid: return "development"
    if doy < late: return "mid-season"
    return "late-season"

D["S"] = ((D.z_evi2 <= PARAMS["z_stress_evi2"]) & ((D.z_ndmi <= PARAMS["z_corrob_water"]) | (D.z_dvv <= PARAMS["z_corrob_water"]))).astype(int)
D["S_unattr"] = ((D.z_evi2 <= PARAMS["z_stress_evi2"]) & ~((D.z_ndmi <= PARAMS["z_corrob_water"]) | (D.z_dvv <= PARAMS["z_corrob_water"]))).astype(int)
D["R"] = (D.z_evi2 >= PARAMS["z_recovery"]).astype(int)
D.loc[D.z_evi2.isna(), ["S", "R"]] = 0

# cell-level leave-one-out synchrony among cultivated parcels (guard makes this cell safe to re-run)
D = D.drop(columns=[c for c in D.columns if c.startswith("cultivated") or c in ("cell_S", "cell_n", "synchrony")], errors="ignore")
D = D.merge(PY[["parcel_id", "year", "cultivated"]], on=["parcel_id", "year"], how="left")
cult = D[D.cultivated == 1]
g = cult.groupby(["cell", "year", "cidx"]).S.agg(["sum", "count"]).rename(columns={"sum": "cell_S", "count": "cell_n"}).reset_index()
D = D.merge(g, on=["cell", "year", "cidx"], how="left")
D["synchrony"] = ((D.cell_S - D.S) / (D.cell_n - 1).clip(lower=1)).where(D.cell_n > 1)

# --- vectorised episode state machine over the 23-composite grid -----------------------------------
T = 23
def extract_episodes(D, s_col="S", r_col="R", z_col="z_evi2"):
    """Run the Tip–Tap state machine for every cultivated parcel-year. Returns one row per episode."""
    key = D[["parcel_id", "year"]].drop_duplicates().reset_index(drop=True); key["row"] = np.arange(len(key))
    Dk = D.merge(key, on=["parcel_id", "year"])
    def mat(col, fill=np.nan):
        M = np.full((len(key), T), fill, dtype=float); M[Dk.row.values, Dk.cidx.values] = Dk[col].values; return M
    S, R, Z, ZN, ZV, SYN = mat(s_col, 0), mat(r_col, 0), mat(z_col), mat("z_ndmi"), mat("z_dvv"), mat("synchrony")
    info = key.merge(PY[["parcel_id", "year", "crop", "cultivated"]], on=["parcel_id", "year"], how="left")
    win = np.array([crop_window(c) for c in info.crop]); w0, w1 = win[:, 0], win[:, 1]
    active = info.cultivated.fillna(0).values == 1
    in_ep = np.zeros(len(key), bool); onset = np.full(len(key), -1); rows = []
    def emit(idx, end_t, recovered):
        for r in idx:
            o = onset[r]; seg = Z[r, o:end_t + 1]
            pre = S[r, max(o - PARAMS["pre_onset_clear"], w0[r]):o]
            chronic = (o < w0[r] + PARAMS["pre_onset_clear"]) or (pre.sum() > 0)
            rows.append((r, o, end_t, recovered, np.nanmin(seg), Z[r, o], np.nansum(np.clip(PARAMS["z_stress_evi2"] - seg, 0, None)),
                         chronic, SYN[r, o], ZN[r, max(o - 1, 0)], ZV[r, max(o - 1, 0)], Z[r, max(o - 1, 0)]))
    for t in range(T - 1):
        inwin = active & (t >= w0) & (t + 1 <= w1)
        rec = in_ep & inwin & (t >= onset + PARAMS["min_persist"]) & (R[:, t] == 1) & (R[:, t + 1] == 1)
        emit(np.where(rec)[0], t, 1); in_ep[rec] = False
        start = ~in_ep & inwin & (S[:, t] == 1) & (S[:, t + 1] == 1)
        onset[start] = t; in_ep[start] = True
        closing = in_ep & (t + 1 == w1)                      # harvest composite reached -> right-censor
        emit(np.where(closing)[0], t + 1, 0); in_ep[closing] = False
    emit(np.where(in_ep)[0], T - 1, 0)
    E = pd.DataFrame(rows, columns=["row", "onset_cidx", "end_cidx", "recovered", "min_z", "z_onset", "burden", "chronic",
                                    "synchrony_onset", "z_ndmi_pre", "z_dvv_pre", "z_evi2_pre"])
    E = E.merge(key, on="row").drop(columns="row")
    E["dur_d"] = (E.end_cidx - E.onset_cidx) * PARAMS["composite_days"]
    E["acute"] = (~E.chronic.astype(bool)).astype(int)
    E = E.merge(PY[["parcel_id", "year", "crop", "county", "cell", "z_int"]], on=["parcel_id", "year"], how="left")
    E["onset_doy"] = cidx_to_doy(E.onset_cidx); E["onset_stage"] = [onset_stage(c, d) for c, d in zip(E.crop, E.onset_doy)]
    E["mechanism"] = np.select([E.chronic.astype(bool), E.synchrony_onset >= 0.5,
                                (E.z_ndmi_pre <= PARAMS["z_corrob_water"]) | (E.z_dvv_pre <= PARAMS["z_corrob_water"])],
                               ["chronic", "synchronous", "water-led"], default="idiosyncratic")
    return E, int(active.sum())

E, n_py = extract_episodes(D)
E = E.merge(PARCELS.drop(columns="geometry"), on="parcel_id", how="left", suffixes=("", "_p"))
E = E.merge(PMLI[["parcel_id", "year", "mulch"]], on=["parcel_id", "year"], how="left")
_pm_mean = PMLI.groupby("parcel_id").mulch.mean().round() if len(PMLI) else pd.Series(dtype=float)
E["mulch"] = E.mulch.fillna(E.parcel_id.map(_pm_mean)).fillna(0).astype(int)

# --- block-level demand covariate --------------------------------------------------------------------
if PATHS["era5_block"].exists():
    era = pd.read_csv(PATHS["era5_block"]); era["grid_start"] = pd.to_datetime(era.grid_start)
    era["year"] = era.grid_start.dt.year; era["cidx"] = ((era.grid_start - pd.to_datetime(era.year.astype(str) + "-03-01")).dt.days // 12).astype(int)
    for v in ["vpd", "tmax", "et0"]:
        era[f"z_{v}"] = era.groupby(["cell", "cidx"])[v].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))
    era3 = era.sort_values(["cell", "year", "cidx"]); era3["z_vpd3"] = era3.groupby(["cell", "year"]).z_vpd.transform(lambda s: s.rolling(3, min_periods=1).mean())
    E = E.merge(era3[["cell", "year", "cidx", "z_vpd3", "z_tmax", "z_et0"]].rename(columns={"cidx": "onset_cidx"}), on=["cell", "year", "onset_cidx"], how="left")
    E["demand"] = E.z_vpd3; DEMAND_SOURCE = "ERA5-Land VPD anomaly (3-composite mean at onset)"
else:
    E["demand"] = E.synchrony_onset; DEMAND_SOURCE = "EO leave-one-out cell synchrony at onset (ERA5 export absent)"
E["high_demand"] = (E.demand >= E.demand.quantile(0.75)).astype(int)
MANIFEST["demand_covariate"] = DEMAND_SOURCE
E["is_test"] = E.year.isin(PARAMS["test_years"]).astype(int)
E.to_parquet(OUT / "interim" / f"episodes_{tag}.parquet", index=False)

print(f"cultivated parcel-years: {n_py:,} | episodes: {len(E):,} ({len(E)/n_py:.2f} per cultivated parcel-year) | recovered before harvest: {E.recovered.mean():.1%} | acute share: {E.acute.mean():.1%}")
print("mechanism mix:", E.mechanism.value_counts(normalize=True).round(3).to_dict()); print("demand covariate:", DEMAND_SOURCE)
display(E.groupby("year").agg(episodes=("dur_d", "size"), recovered=("recovered", "mean"), median_dur_d=("dur_d", "median"), acute=("acute", "mean")).round(2))


## 1.10 · Who recovers, and how fast? (Fig. 5, Table 3)

Kaplan–Meier estimates of the probability that a stress episode is still open $t$ days after onset, with harvest as right-censoring. $1 - S(t)$ is the empirical recovery probability, the descriptive form of the paper's central quantity $P_{rec}(t \mid \cdot)$. Stratified by crop, onset stage, mechanism, parcel size, oasis position and irrigation technology. Log-rank tests are reported for each stratification.

In [ ]:
from lifelines.statistics import multivariate_logrank_test
E["fringe_tercile"] = E.fringe_tercile.astype(str); E["size_class"] = E.size_class.astype(str)
E["mulch_lbl"] = E.mulch.map({1: "mulched (drip-like)", 0: "unmulched (surface)"})
E["t"] = E.dur_d.clip(lower=PARAMS["composite_days"])

KM_COLORS = ["#00B050", "#FF0000", "#0070C0", "#ED7D31"]
KM_LABELS = {"unmulched (surface)": "surface irrigated", "mulched (drip-like)": "mulched drip"}

def km_panel(ax, df, by, letter, heading, order, ylabel=True):
    groups = [g for g in order if (df[by] == g).sum() >= 40]
    for g, col in zip(groups, KM_COLORS):
        sub = df[df[by] == g]
        KaplanMeierFitter().fit(sub.t, sub.recovered, label=KM_LABELS.get(g, g)).plot_survival_function(ax=ax, ci_show=False, lw=2, color=col)
    d = df[df[by].isin(groups)]
    p = multivariate_logrank_test(d.t, d[by], d.recovered).p_value
    print(by, f"log-rank p = {p:.2g}")
    ax.text(0.95, 0.55, "log-rank p < 0.001" if p < 0.001 else f"log-rank p = {p:.3f}", transform=ax.transAxes, ha="right", va="center", fontsize=9, color="0.35")
    ax.axvline(36, color="0.5", lw=0.8, ls=":")
    ax.text(0.04, 0.95, f"({letter})", transform=ax.transAxes, ha="left", va="top", fontsize=10, fontweight="bold")
    ax.set_title(heading, fontsize=11)
    ax.set_xlim(0, 190); ax.set_ylim(0, 1); ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels(["0.00", "0.25", "0.50", "0.75", "1.00"])
    ax.set_xlabel("days since onset"); ax.set_ylabel("P(episode still open)" if ylabel else "")
    ax.legend(loc="upper right", fontsize=9, frameon=False, handlelength=1.8)

fig, axes = plt.subplots(2, 3, figsize=(11, 6.6)); axes = axes.ravel()
km_panel(axes[0], E, "crop", "a", "Crop", ["cotton", "orchard", "winter wheat", "other economic"])
km_panel(axes[1], E[E.acute == 1], "onset_stage", "b", "Onset stage (acute episodes)", ["development", "mid-season", "late-season"], ylabel=False)
km_panel(axes[2], E, "mechanism", "c", "Mechanism fingerprint", ["water-led", "synchronous", "idiosyncratic", "chronic"], ylabel=False)
km_panel(axes[3], E, "size_class", "d", "Parcel size", ["0.5–2 ha", "2–10 ha", ">10 ha"])
km_panel(axes[4], E, "fringe_tercile", "e", "Oasis position (within county)", ["core", "middle", "fringe"], ylabel=False)
km_panel(axes[5], E, "mulch_lbl", "f", "Irrigation technology", ["unmulched (surface)", "mulched (drip-like)"], ylabel=False)
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig2_km_recovery.png", bbox_inches="tight"); plt.show()

# recovery probability at fixed horizons, by stratum -> Table 1
def rec_at(df, horizons=PARAMS["recovery_horizons_d"]):
    km = KaplanMeierFitter().fit(df.t, df.recovered)
    return pd.Series({f"P_rec_{h}d": float(1 - km.predict(h)) for h in horizons} | {"n": len(df), "median_open_d": km.median_survival_time_})
T1 = pd.concat({k: E.groupby(k).apply(rec_at) for k in ["crop", "onset_stage", "mechanism", "size_class", "fringe_tercile", "mulch_lbl", "year"]}, names=["stratum", "level"])
T1.to_csv(OUT / "tables" / "table1_recovery_by_stratum.csv"); display(T1.round(3))

## 1.11 · Recovery model and Recovery Capacity Index

A Cox proportional-hazards model of time-to-recovery (event = Tap; censoring = harvest) is fitted on the training seasons (2019–2022) with **only information available at onset**: crop, onset stage, severity at onset, mechanism fingerprint, block-level demand, year, and the structural covariates of 1.4. Standard errors are clustered on the 9 km ERA5 cell so that parcels sharing the same weather are not counted as independent evidence. Hazard ratios > 1 mean *faster recovery*.

Two products come out of the fitted model:

* **Episode-level $P_{rec}(h)$** at 24, 36 and 48 days for every episode, including the held-out 2023–2025 seasons (forward validation in 1.15).
* **Structural Recovery Capacity Index (RCI)** for every parcel: the predicted probability of recovering within 36 days for a *reference episode* (cotton, development-stage onset, acute, water-led, median demand, reference year) given only the parcel's structural covariates. RCI isolates *where the landscape is resilient* from *what happened in a given year*, which is what a targeting map needs.

In [ ]:
E["cell_code"] = pd.factorize(E.cell)[0]

NUM = ["log_area"] + HYD_NUM + ["hyd_unmapped", "log_dist_desert", "mulch", "log_ntl", "dist_settlement_km", "dist_road_km", "z_onset", "demand"]
CAT = {"crop": "cotton", "onset_stage": "development", "mechanism": "water-led", "salinity_class": "mid", "year": 2021}
def design(df):
    X = df[NUM].copy()
    for c, ref in CAT.items():
        d = pd.get_dummies(df[c].astype(str), prefix=c, dtype=float).drop(columns=[f"{c}_{ref}"], errors="ignore")
        X = pd.concat([X, d], axis=1)
    return X.fillna(0)
Em = E[E.crop != "unclassified"].copy()
Xall = design(Em); TRAIN = Em.year.isin(PARAMS["baseline_years"]).values
Xtr = Xall[TRAIN].copy(); Xtr["t"] = Em.t[TRAIN].values; Xtr["recovered"] = Em.recovered[TRAIN].values; Xtr["cell_code"] = Em.cell_code[TRAIN].values
Xtr = Xtr.loc[:, (Xtr.std() > 0) | Xtr.columns.isin(["t", "recovered", "cell_code"])]
cph = CoxPHFitter(penalizer=0.01)
cph.fit(Xtr, duration_col="t", event_col="recovered", cluster_col="cell_code", robust=True)
Xte = Xall[~TRAIN][ [c for c in Xtr.columns if c not in ("t", "recovered", "cell_code")] ]
Xtr_f = Xtr.drop(columns=["t", "recovered", "cell_code"])
c_train = concordance_index(Em.t[TRAIN], -cph.predict_partial_hazard(Xtr_f), Em.recovered[TRAIN])
c_test  = concordance_index(Em.t[~TRAIN], -cph.predict_partial_hazard(Xte), Em.recovered[~TRAIN])
print(f"Cox recovery model: {TRAIN.sum():,} training episodes, {(~TRAIN).sum():,} held-out (2023–2025); concordance train {c_train:.3f} | forward test {c_test:.3f}")
HR = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].rename(columns={"exp(coef)": "HR", "exp(coef) lower 95%": "lo", "exp(coef) upper 95%": "hi"})
HR.to_csv(OUT / "tables" / "table2_cox_hazard_ratios.csv"); display(HR.round(3))

# episode-level recovery probabilities at fixed horizons (train and test)
Xf = Xall[Xtr_f.columns]
sf = cph.predict_survival_function(Xf, times=PARAMS["recovery_horizons_d"])
for h in PARAMS["recovery_horizons_d"]:
    Em[f"p_rec_{h}d"] = 1 - sf.loc[h].values
E = E.drop(columns=[c for c in E.columns if c.startswith("p_rec_")], errors="ignore")
E = E.merge(Em[["parcel_id", "year", "onset_cidx"] + [f"p_rec_{h}d" for h in PARAMS["recovery_horizons_d"]]], on=["parcel_id", "year", "onset_cidx"], how="left")

# structural RCI: reference episode, parcel covariates only
ref = PARCELS[["parcel_id", "log_area"] + HYD_NUM + ["hyd_unmapped", "log_dist_desert", "log_ntl", "dist_settlement_km", "dist_road_km", "salinity_class"]].copy()
ref["mulch"] = ref.parcel_id.map(PMLI.groupby("parcel_id").mulch.mean().round()).fillna(0)
ref["z_onset"] = float(Em.z_onset[TRAIN].median()); ref["demand"] = float(Em.demand[TRAIN].median()); ref["acute"] = 1  # kept for the GLM in §9
for c, r in CAT.items():
    if c != "salinity_class": ref[c] = r
Xref = design(ref).reindex(columns=Xtr_f.columns, fill_value=0)
PARCELS["rci"] = 1 - cph.predict_survival_function(Xref, times=[PARAMS["rci_horizon_d"]]).loc[PARAMS["rci_horizon_d"]].values
print(f"structural RCI (P_rec within {PARAMS['rci_horizon_d']} d, reference episode): median {PARCELS.rci.median():.2f}, IQR {PARCELS.rci.quantile(.25):.2f}–{PARCELS.rci.quantile(.75):.2f}")

Em["y36"] = ((Em.recovered == 1) & (Em.t <= 36)).astype(int)
MANIFEST["models"] = dict(cox_c_train=c_train, cox_c_test=c_test)


## 1.12 · Structural differences in recovery (Fig. 6)

In [ ]:
Eg = E[(E.crop != "unclassified") & ((E.t >= 36) | (E.recovered == 1))].copy()
Eg["y36"] = ((Eg.recovered == 1) & (Eg.t <= 36)).astype(int)
Eg["fringe_km"] = Eg.dist_desert_km; Eg["channel_km"] = Eg.hyd_remote_km
Eg["period"] = np.where(Eg.year.isin(PARAMS["baseline_years"]), "2019–22", "2023–25")
F_BASE = f"y36 ~ log_area + log_dist_desert + {HYD_TERMS} + hyd_unmapped + mulch + C(salinity_class, Treatment('mid')) + log_ntl + dist_settlement_km + dist_road_km + z_onset + demand + C(crop) + C(onset_stage) + C(mechanism) + C(year)"
F_INT  = F_BASE + " + log_dist_desert:high_demand + hyd_rank:high_demand + high_demand"
GLM_COLS = ["y36", "log_area", "log_dist_desert"] + HYD_NUM + ["hyd_rank_t12", "hyd_unmapped", "mulch", "salinity_class", "log_ntl", "dist_settlement_km", "dist_road_km",
            "z_onset", "acute", "demand", "high_demand", "crop", "onset_stage", "mechanism", "year", "cell_code", "dist_desert_km", "hyd_remote_km"]
Eg = Eg.dropna(subset=GLM_COLS).reset_index(drop=True)
def fit_glm(formula, df):
    return smf.glm(formula, data=df, family=sm.families.Binomial()).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(df.cell_code)[0]})
m1 = fit_glm(F_BASE, Eg); m2 = fit_glm(F_INT, Eg)
me = m1.get_margeff(at="overall").summary_frame()
me = me.loc[[i for i in me.index if not i.startswith("C(")]].rename(columns={"dy/dx": "AME", "Pr(>|z|)": "p"})
me["AME_pp"] = 100 * me["AME"]
print("H1 — average marginal effects on P(recovered within 36 d), percentage points:"); display(me[["AME_pp", "p"]].round(3))
inter = m2.summary2().tables[1].loc[["log_dist_desert:high_demand", "hyd_rank:high_demand", "high_demand"], ["Coef.", "Std.Err.", "P>|z|"]]
print("\nH2 — position × high-demand interactions (log-odds):"); display(inter.round(3))
# tier-2 sensitivity: does the head-tail effect survive when undirected (tier-2) ranks are added?
m1_t12 = fit_glm(F_BASE.replace("hyd_rank", "hyd_rank_t12"), Eg)
print(f"H1 head-tail (hyd_rank) log-odds — tier 1 only: {m1.params['hyd_rank']:+.3f} (p={m1.pvalues['hyd_rank']:.3f}) | tiers 1+2: {m1_t12.params['hyd_rank_t12']:+.3f} (p={m1_t12.pvalues['hyd_rank_t12']:.3f})")
MANIFEST["H1_hyd_rank_tier_sensitivity"] = dict(tier1=float(m1.params["hyd_rank"]), tier12=float(m1_t12.params["hyd_rank_t12"]))
me.to_csv(OUT / "tables" / "table4_H1_marginal_effects.csv"); inter.to_csv(OUT / "tables" / "table5_H2_interactions.csv")
MANIFEST["H1_H2"] = dict(n=int(len(Eg)), ame=me["AME_pp"].round(2).to_dict(), interactions=inter["Coef."].round(3).to_dict(), interaction_p=inter["P>|z|"].round(3).to_dict())

# predicted recovery by oasis position x demand (H2 visual) — Fig. 3b
grid = Eg.copy(); rows = []
for hd in [0, 1]:
    for q in np.linspace(Eg.dist_desert_km.quantile(.05), Eg.dist_desert_km.quantile(.95), 25):
        g = grid.copy(); g["high_demand"] = hd; g["log_dist_desert"] = np.log1p(q)
        rows.append((hd, q, m2.predict(g).mean()))
PRED = pd.DataFrame(rows, columns=["high_demand", "fringe_km", "p_rec36"])

# Lorenz / concentration of non-recovered stress-days
E["nonrec_days"] = np.where(E.recovered == 0, E.dur_d, 0)
PB = E.groupby("parcel_id").agg(nonrec_days=("nonrec_days", "sum"), episodes=("dur_d", "size"), nonrec_eps=("recovered", lambda s: (1 - s).sum())).reset_index()
PB = PB.merge(PARCELS[["parcel_id", "area_ha", "dist_desert_km", "hyd_remote_km", "hyd_rank", "rci", "county", "cell", "size_class", "fringe_tercile"]], on="parcel_id", how="right").fillna({"nonrec_days": 0, "episodes": 0, "nonrec_eps": 0})
def lorenz(df, order_col, weight_col="nonrec_days", ascending=True):
    d = df.sort_values(order_col, ascending=ascending)
    x = np.cumsum(np.ones(len(d))) / len(d); y = np.cumsum(d[weight_col].values) / max(d[weight_col].sum(), 1e-9)
    ci = 1 - 2 * (np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(y, x)          # concentration index: >0 burden concentrated in low values of order_col
    return x, y, ci
Lz = {k: lorenz(PB, c, ascending=asc) for k, (c, asc) in {"parcel size (small→large)": ("area_ha", True), "oasis position (fringe→core)": ("dist_desert_km", True), "head→tail rank (tier 1)": ("hyd_rank", True)}.items()}
MANIFEST["concentration_index_nonrecovered_days"] = {k: float(v[2]) for k, v in Lz.items()}
print("\nConcentration index of non-recovered stress-days (positive = concentrated at the start of the ordering):", {k: round(v[2], 3) for k, v in Lz.items()})

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
ax = axes[0]; sub = me.sort_values("AME_pp")
ax.barh(sub.index, sub.AME_pp, xerr=100 * 1.96 * sub["Std. Err."], color=np.where(sub.AME_pp > 0, "#0ee66f", "#ee330a"), alpha=.85)
ax.axvline(0, color="k", lw=.8); ax.set_xlabel("effect on P(recover ≤ 36 d), percentage points"); ax.set_title("a", loc="left", fontweight="bold")
ax = axes[1]
for hd, lab, c in [(0, "normal demand", "#0c1ee3"), (1, "high demand (top quartile)", "#ee330a")]:
    p = PRED[PRED.high_demand == hd]; ax.plot(p.fringe_km, p.p_rec36, lw=2, color=c, label=lab)
ax.set_xlabel("distance from oasis fringe (km)"); ax.set_ylabel("predicted P(recover ≤ 36 d)"); ax.legend(frameon=False, fontsize=10)
print(f"interaction p = {inter.loc['log_dist_desert:high_demand','P>|z|']:.2g}"); ax.set_title("b", loc="left", fontweight="bold")
ax = axes[2]
for (k, (x, y, ci)), c in zip(Lz.items(), ["#0c1ee3", "#ee330a", "#0ee66f"]):
    ax.plot(x, y, lw=2, color=c, label=f"{k}  CI={ci:+.2f}")
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("cumulative share of parcels"); ax.set_ylabel("cumulative share of non-recovered stress-days"); ax.legend(frameon=False, fontsize=7.5)
ax.set_title("c", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig3_inequality.png", bbox_inches="tight"); plt.show()


## 1.13 · Recovery deficit and subsequent land-use change (Fig. 7)

In [ ]:
base_y, test_y = PARAMS["baseline_years"], PARAMS["test_years"]
pyb = PY[PY.year.isin(base_y)].groupby("parcel_id").agg(cult_base=("cultivated", "sum"), zint_base=("z_int", "mean"))
pyt = PY[PY.year.isin(test_y)].groupby("parcel_id").agg(cult_test=("cultivated", "sum"), n_test=("cultivated", "size"), zint_test=("z_int", "mean"))
Eb = E[E.year.isin(base_y)].groupby("parcel_id").agg(eps_base=("dur_d", "size"), nonrec_share=("recovered", lambda s: 1 - s.mean()), nonrec_days_base=("nonrec_days", "sum"), minz_base=("min_z", "mean"))
X3 = PARCELS.drop(columns="geometry").merge(pyb, on="parcel_id").merge(pyt, on="parcel_id", how="left").merge(Eb, on="parcel_id", how="left")
X3 = X3[X3.cult_base >= 3].copy()
X3["exposure"] = X3.eps_base.fillna(0) / X3.cult_base
X3["deficit_days"] = X3.nonrec_days_base.fillna(0) / X3.cult_base
X3["nonrec_share"] = X3.nonrec_share.fillna(0); X3["had_episode"] = (X3.eps_base.fillna(0) > 0).astype(int)
X3["exit"] = ((X3.n_test - X3.cult_test) >= 1).astype(int)
c19 = CROP[CROP.year == base_y[0]].set_index("parcel_id").crop; c24 = CROP[CROP.year == max(y for y in CROP.year.unique() if y <= test_y[-1])].set_index("parcel_id").crop
X3["orchard_conv"] = ((c19.reindex(X3.parcel_id).values != "orchard") & (c24.reindex(X3.parcel_id).values == "orchard")).astype(int)
X3["cell_code"] = pd.factorize(X3.cell)[0]
# cross-check: EO fallow vs crop-map 'unclassified' in the same year
chk = PY.merge(CROP[["parcel_id", "year", "crop"]], on=["parcel_id", "year"], suffixes=("", "_map"))
agree = pd.crosstab(chk.cultivated == 0, chk.crop_map == "unclassified", normalize="index")
MANIFEST["fallow_vs_cropmap_unclassified_rate"] = dict(fallow=float(agree.loc[True, True]) if True in agree.index else np.nan, cultivated=float(agree.loc[False, True]))
print(f"cross-check: share of parcel-years mapped 'unclassified' when EO says fallow {MANIFEST['fallow_vs_cropmap_unclassified_rate']['fallow']:.2f} vs cultivated {MANIFEST['fallow_vs_cropmap_unclassified_rate']['cultivated']:.2f}")
print(f"parcels in exit analysis: {len(X3):,} | exit rate 2023–25: {X3.exit.mean():.1%} | annual→orchard conversion: {X3.orchard_conv.mean():.1%}")
F3 = "{y} ~ deficit_days + exposure + nonrec_share + log_area + log_dist_desert + " + HYD_TERMS + " + hyd_unmapped + C(salinity_class, Treatment('mid')) + log_ntl + dist_settlement_km + dist_road_km + C(county)"
res3 = {}
for y in ["exit", "orchard_conv"]:
    if X3[y].sum() < 20: print(f"{y}: too few events ({X3[y].sum()}) for a model in this sample"); continue
    X3m = X3.dropna(subset=["deficit_days", "exposure", "nonrec_share", "log_area", "log_dist_desert"] + HYD_NUM + [ "log_ntl", "dist_settlement_km", "dist_road_km", "salinity_class", "county"]).copy()
    m = smf.logit(F3.format(y=y), data=X3m).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(X3m.cell_code)[0]}, maxiter=300)
    tab = pd.DataFrame({"OR": np.exp(m.params), "lo": np.exp(m.conf_int()[0]), "hi": np.exp(m.conf_int()[1]), "p": m.pvalues})
    tab = tab.loc[[i for i in tab.index if not i.startswith(("C(county", "Intercept"))]]; res3[y] = tab
    print(f"\nH3 — {y} (odds ratios, county FE, cell-clustered SE), n={len(X3):,}, events={int(X3[y].sum())}"); display(tab.round(3))
    tab.to_csv(OUT / "tables" / f"table6_H3_{y}.csv")
MANIFEST["H3"] = {k: {"OR_deficit_days": float(v.loc["deficit_days", "OR"]), "p": float(v.loc["deficit_days", "p"])} for k, v in res3.items()}
if res3:
    OR_LABELS = {"C(salinity_class, Treatment('mid'))[T.high]": "salinity: high (vs mid)", "C(salinity_class, Treatment('mid'))[T.low]": "salinity: low (vs mid)",
                 "log_ntl": "log night-time light", "exposure": "exposure (episodes/season)", "nonrec_share": "non-recovered share",
                 "deficit_days": "recovery deficit (per day/season)", "hyd_dz_m": "fall from intake (per m)", "dist_road_km": "road distance (per km)",
                 "dist_settlement_km": "settlement distance (per km)", "hyd_rank": "head–tail rank", "hyd_unmapped": "unmapped channel",
                 "log_dist_desert": "log fringe distance", "log_area": "log parcel area"}
    order = list(next(iter(res3.values())).sort_values("OR").index)
    ypos = {v: i for i, v in enumerate(order)}
    fig, axes = plt.subplots(1, len(res3), figsize=(5.2 * len(res3), 5), sharey=True, squeeze=False)
    for ax, lab, (y, tab) in zip(axes[0], "ab", res3.items()):
        for v, r in tab.reindex(order).dropna(subset=["OR"]).iterrows():
            col = "#FF0000" if r.lo > 1 else "#00B050" if r.hi < 1 else "black"
            ax.plot([r.lo, r.hi], [ypos[v]] * 2, color=col, lw=1.6, solid_capstyle="butt")
            ax.plot(r.OR, ypos[v], "o", color=col, ms=5, mec="white", mew=0.6, zorder=3)
        ax.axvline(1, color="k", lw=.8)
        ax.set_xscale("log"); ax.set_xlim(0.25, 3.3)
        ax.set_xticks([0.3, 0.5, 1, 2, 3]); ax.set_xticklabels(["0.3", "0.5", "1", "2", "3"]); ax.minorticks_off()
        ax.set_yticks(range(len(order))); ax.set_yticklabels([OR_LABELS.get(v, v) for v in order]); ax.set_ylim(-0.6, len(order) - 0.4)
        ax.tick_params(axis="y", length=0)
        ax.set_xlabel("odds ratio (log scale)")
        ax.set_title(lab, loc="left", fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT / "figures" / "fig4_exit.png", bbox_inches="tight"); plt.show()

## 1.14 · Targeting a fixed budget at parcel scale (Fig. 8)

A water manager or a development programme can reach a fraction of parcels. Which 2019–2022 signal best predicts where *unrecovered* stress will concentrate in 2023–2025? Five ranking rules are compared out of sample: **exposure** (episodes per season — the early-warning logic: go where stress is detected most often), **recovery deficit** (non-recovered stress-days per season), **structural RCI** (the model-based map from 1.11; lowest capacity first), **smallest parcels first** (an equity rule), and **random**. Gain curves show the share of 2023–2025 non-recovered stress-days captured against the share of parcels (or of area) treated. Because who is reached matters as much as how much, the composition of the targeted set (small parcels, fringe parcels) is reported for each rule at a 20 % budget.

In [ ]:
Et = E[E.year.isin(PARAMS["test_years"])].groupby("parcel_id").agg(nonrec_days_test=("nonrec_days", "sum"), nonrec_eps_test=("recovered", lambda s: int((1 - s).sum()))).reset_index()
TG = X3[["parcel_id", "area_ha", "exposure", "deficit_days", "nonrec_share", "size_class", "fringe_tercile", "county"]].merge(PARCELS[["parcel_id", "rci"]], on="parcel_id").merge(Et, on="parcel_id", how="left").fillna({"nonrec_days_test": 0, "nonrec_eps_test": 0})
rng = np.random.default_rng(PARAMS["seed"]); TG["random"] = rng.random(len(TG)); TG["small_first"] = -TG.area_ha; TG["rci_deficit"] = 1 - TG.rci
RULES = {"exposure (early-warning logic)": "exposure", "recovery deficit (2019–22)": "deficit_days", "structural RCI (lowest first)": "rci_deficit", "smallest parcels first": "small_first", "random": "random"}
def gain(df, score, outcome="nonrec_days_test", weight=None):
    d = df.sort_values(score, ascending=False, kind="mergesort")
    x = np.cumsum(d[weight].values) / d[weight].sum() if weight else np.arange(1, len(d) + 1) / len(d)
    y = np.cumsum(d[outcome].values) / max(d[outcome].sum(), 1e-9)
    return x, y
def capture_at(df, score, budget, outcome="nonrec_days_test"):
    x, y = gain(df, score, outcome); return float(np.interp(budget, x, y))
rows = []
for lab, sc in RULES.items():
    top = TG.sort_values(sc, ascending=False).head(int(0.2 * len(TG)))
    rows.append(dict(rule=lab, capture_10pct=capture_at(TG, sc, .10), capture_20pct=capture_at(TG, sc, .20), capture_30pct=capture_at(TG, sc, .30),
                     share_small_in_top20=float(top.size_class.astype(str).isin(["<0.5 ha", "0.5–2 ha"]).mean()), share_fringe_in_top20=float((top.fringe_tercile.astype(str) == "fringe").mean()),
                     area_share_top20=float(top.area_ha.sum() / TG.area_ha.sum())))
T7 = pd.DataFrame(rows).set_index("rule"); T7.to_csv(OUT / "tables" / "table7_targeting.csv"); display(T7.round(3))
MANIFEST["H4_targeting"] = T7[["capture_20pct"]].round(3).capture_20pct.to_dict()
uplift = T7.loc["recovery deficit (2019–22)", "capture_20pct"] / max(T7.loc["exposure (early-warning logic)", "capture_20pct"], 1e-9)
print(f"At a 20% parcel budget, ranking by recovery deficit captures {T7.loc['recovery deficit (2019–22)','capture_20pct']:.0%} of 2023–25 non-recovered stress-days vs {T7.loc['exposure (early-warning logic)','capture_20pct']:.0%} by exposure (×{uplift:.2f}); random ≈ 20%.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for (lab, sc), c in zip(RULES.items(), ["#e63811", "#089b4a", "#0b62e5", "#e4a410", "#888"]):
    x, y = gain(TG, sc); axes[0].plot(x, y, lw=2, color=c, label=lab)
    x, y = gain(TG, sc, weight="area_ha"); axes[1].plot(x, y, lw=2, color=c, label=lab)
for ax, xl in zip(axes, ["share of parcels treated", "share of cropland area treated"]):
    ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel(xl); ax.set_ylabel("share of 2023–25 non-recovered stress-days captured"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[0].legend(frameon=False, fontsize=10); axes[0].set_title("a", loc="left", fontweight="bold"); axes[1].set_title("b", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig5_targeting.png", bbox_inches="tight"); plt.show()


## 1.15 · Validation and sensitivity (Table 2)

In [ ]:
VAL = {}
# (ii) leave-one-county-out concordance
loco = {}
for cty in Em.county.unique():
    tr = Em.county != cty
    if (~tr).sum() < 100 or Em.recovered[~tr].sum() < 20: continue
    Xc = Xall.loc[tr.values, Xtr_f.columns].copy(); Xc["t"] = Em.t[tr].values; Xc["recovered"] = Em.recovered[tr].values
    Xc = Xc.loc[:, (Xc.std() > 0)]
    try:
        m = CoxPHFitter(penalizer=0.05).fit(Xc, "t", "recovered")
        loco[cty] = concordance_index(Em.t[~tr], -m.predict_partial_hazard(Xall.loc[(~tr).values, Xc.columns.drop(["t", "recovered"])]), Em.recovered[~tr])
    except Exception as ex: loco[cty] = np.nan
VAL["leave_one_county_out_concordance"] = {k: round(float(v), 3) for k, v in loco.items()}

# (iii) placebo 1: shuffled oasis position within county -> the H2 interaction should vanish
Ep = Eg.copy(); Ep["log_dist_desert"] = Ep.groupby("county").log_dist_desert.transform(lambda s: s.sample(frac=1, random_state=PARAMS["seed"]).values)
mp = fit_glm(F_INT, Ep); VAL["placebo_shuffled_position_interaction"] = dict(coef=float(mp.params["log_dist_desert:high_demand"]), p=float(mp.pvalues["log_dist_desert:high_demand"]))
del Ep

# (iv) placebo 2: pre-season (Mar–Apr) EVI2 anomaly 2019–22 should not predict 2023–25 exit once cultivation is controlled
pre = D[(D.cidx <= 3) & D.year.isin(PARAMS["baseline_years"])].groupby("parcel_id").z_evi2.mean().rename("z_preseason")
X3p = X3.merge(pre, on="parcel_id", how="left").dropna(subset=["z_preseason"])
# county fixed effects are unidentified in counties with no exits (or only exits): drop them from the placebo fit
ok_cty = X3p.groupby("county").exit.agg(["sum", "count"]).query("sum > 0 and sum < count").index
X3p = X3p[X3p.county.isin(ok_cty)].copy()
if X3p.exit.sum() >= 20:
    try:
        mpl = smf.logit("exit ~ z_preseason + deficit_days + log_area + C(county)", data=X3p).fit(
            disp=0, method="bfgs", maxiter=500, cov_type="cluster", cov_kwds={"groups": pd.factorize(X3p.cell_code)[0]})
    except Exception:
        mpl = smf.glm("exit ~ z_preseason + deficit_days + log_area + C(county)", data=X3p, family=sm.families.Binomial()).fit(
            cov_type="cluster", cov_kwds={"groups": pd.factorize(X3p.cell_code)[0]})
    VAL["placebo_preseason_anomaly_on_exit"] = dict(OR_preseason=float(np.exp(mpl.params["z_preseason"])), p_preseason=float(mpl.pvalues["z_preseason"]),
                                                    OR_deficit=float(np.exp(mpl.params["deficit_days"])), p_deficit=float(mpl.pvalues["deficit_days"]),
                                                    n=int(len(X3p)), counties_dropped=int(X3.county.nunique() - len(ok_cty)))

# (v) definition sensitivity: rerun the episode machine under alternative rules.
#     Only the seven columns the state machine reads are copied (the full 19 M-row frame would need ~5 GB per run).
SLIM = ["parcel_id", "year", "cidx", "z_evi2", "z_ndmi", "z_dvv", "synchrony"]
def sens_run(label, s_mask, r_mask, z=None):
    Dx = D[SLIM].copy()
    if z is not None: Dx["z_evi2"] = z.values                      # parcel-own variant supplies its own anomaly
    Dx["S"] = s_mask.values.astype(np.int8); Dx["R"] = r_mask.values.astype(np.int8); Dx.loc[Dx.z_evi2.isna(), ["S", "R"]] = 0
    Ex, npy = extract_episodes(Dx); del Dx
    Ex = Ex.merge(PARCELS[["parcel_id", "log_dist_desert", "log_area", "dist_settlement_km", "salinity_class"]], on="parcel_id", how="left")
    Ex = Ex.merge(PMLI[["parcel_id", "year", "mulch"]], on=["parcel_id", "year"], how="left"); Ex["mulch"] = Ex.mulch.fillna(0)
    Ex["t"] = Ex.dur_d.clip(lower=12); e36 = Ex[(Ex.t >= 36) | (Ex.recovered == 1)].copy(); e36["y36"] = ((e36.recovered == 1) & (e36.t <= 36)).astype(int)
    g = smf.glm("y36 ~ mulch + log_dist_desert + log_area + dist_settlement_km + C(salinity_class, Treatment('mid')) + C(crop) + C(year)", data=e36, family=sm.families.Binomial()).fit()
    me_ = g.get_margeff().summary_frame()
    out = dict(rule=label, episodes=len(Ex), per_cultivated_py=round(len(Ex) / npy, 3), recovered_share=round(float(Ex.recovered.mean()), 3),
               AME_mulch_pp=round(100 * float(me_.loc["mulch", "dy/dx"]), 2),
               AME_salinity_high_pp=round(100 * float(me_.loc["C(salinity_class, Treatment('mid'))[T.high]", "dy/dx"]), 2))
    del Ex, e36, g; return out

zs, zw, zr = PARAMS["z_stress_evi2"], PARAMS["z_corrob_water"], PARAMS["z_recovery"]
water = (D.z_ndmi <= zw) | (D.z_dvv <= zw)
SENS = pd.DataFrame([
    sens_run("baseline rules", (D.z_evi2 <= zs) & water, D.z_evi2 >= zr),
    sens_run("stress z ≤ -0.8", (D.z_evi2 <= -0.8) & water, D.z_evi2 >= zr),
    sens_run("stress z ≤ -1.2", (D.z_evi2 <= -1.2) & water, D.z_evi2 >= zr),
    sens_run("recovery z ≥ -0.3", (D.z_evi2 <= zs) & water, D.z_evi2 >= -0.3),
    sens_run("recovery z ≥ -0.7", (D.z_evi2 <= zs) & water, D.z_evi2 >= -0.7),
    sens_run("NDMI-only corroboration (no Sentinel-1)", (D.z_evi2 <= zs) & (D.z_ndmi <= zw), D.z_evi2 >= zr),
    sens_run("vigour only (no water corroboration)", (D.z_evi2 <= zs), D.z_evi2 >= zr),
]).set_index("rule")

# parcel-own baseline (the R20 alternative): index lookup instead of a 19 M-row merge
own = D[D.year.isin(PARAMS["baseline_years"])].groupby(["parcel_id", "cidx"]).EVI2_mean.agg(["median", lambda s: 1.4826 * (s - s.median()).abs().median()]); own.columns = ["own_med", "own_mad"]
idx = pd.MultiIndex.from_arrays([D.parcel_id.values, D.cidx.values])
z_own = pd.Series((D.EVI2_mean.values - own.own_med.reindex(idx).values) / own.own_mad.replace(0, np.nan).reindex(idx).values, index=D.index)
SENS.loc["parcel-own baseline (R20 alternative)"] = pd.Series(sens_run("parcel-own baseline (R20 alternative)", (z_own <= zs) & water, z_own >= zr, z=z_own)).drop("rule")
del own, idx, z_own, water
SENS.to_csv(OUT / "tables" / "table8_sensitivity.csv"); display(SENS)

# Sentinel-1 cadence note (2022 = Sentinel-1A alone at 12-day repeat)
cad = pd.DataFrame({"with S1": E.groupby("year").size() / PY[PY.cultivated == 1].groupby("year").size()})
VAL["s1_cadence_note"] = "2022 = Sentinel-1A alone (12-day); compare 'NDMI-only corroboration' row of the sensitivity table"
MANIFEST["validation"] = VAL
print(json.dumps(VAL, indent=1, default=float))


## 1.16 · Study system and episode grammar (Fig. 4); GIS export, manifest and results summary

In [ ]:
# --- Fig. 1: study system, RCI map and the episode grammar on real parcels ---------------------------
from matplotlib.patches import Patch, FancyArrowPatch
from matplotlib.lines import Line2D
Pm = PARCELS.merge(PB[["parcel_id", "nonrec_days", "episodes"]], on="parcel_id", how="left").fillna({"nonrec_days": 0, "episodes": 0})
pref = gpd.read_file(PATHS["prefecture"]).to_crs("EPSG:4326")
fig = plt.figure(figsize=(13, 5.2)); gs_ = fig.add_gridspec(2, 3, width_ratios=[1.35, 1, 1])

# (a) RCI map: septile classes, points drawn in random order so no class hides another
ax = fig.add_subplot(gs_[:, 0]); pref.boundary.plot(ax=ax, color="#666", lw=.8)
pts = Pm.copy(); pts["geometry"] = pts.geometry.centroid
ASI_RAMP = ["#B00000", "#FF3300", "#FF8000", "#FFB833", "#FFFF33", "#99E633", "#33CC00"]   # low RCI (red) -> high RCI (green)
brks = np.unique(np.quantile(pts.rci.dropna(), np.linspace(0, 1, len(ASI_RAMP) + 1)))
pts["rci_cls"] = np.clip(np.digitize(pts.rci, brks[1:-1]), 0, len(brks) - 2)
_o = pts.dropna(subset=["rci"]).sample(frac=1, random_state=PARAMS["seed"])
_o.plot(ax=ax, color=[ASI_RAMP[k] for k in _o.rci_cls], markersize=3, linewidth=0)
ax.legend(handles=[Patch(facecolor=ASI_RAMP[k], edgecolor="none", label=f"{brks[k]:.3f} – {brks[k+1]:.3f}") for k in range(len(brks) - 1)][::-1],
          title=f"RCI, P(recover ≤ {PARAMS['rci_horizon_d']} d)", title_fontsize=10, fontsize=9.5, frameon=False,
          loc="lower right", bbox_to_anchor=(1.12, 0.0))
xmin, ymin, xmax, ymax = pref.total_bounds
L = 100 / (111.32 * np.cos(np.deg2rad((ymin + ymax) / 2)))                                       # 100 km in degrees of longitude
x0s, y0s = xmin + 0.02 * (xmax - xmin), ymin + 0.03 * (ymax - ymin)
ax.plot([x0s, x0s + L], [y0s, y0s], color="black", lw=2.2, solid_capstyle="butt")
ax.plot([x0s, x0s + L / 2], [y0s, y0s], color="white", lw=1.2, solid_capstyle="butt")
ax.text(x0s + L / 2, y0s + 0.015 * (ymax - ymin), "100 km", ha="center", va="bottom", fontsize=8)
ax.add_patch(FancyArrowPatch((0.06, 0.80), (0.06, 0.92), transform=ax.transAxes, arrowstyle="-|>", mutation_scale=12, lw=1.4, color="black"))
ax.text(0.06, 0.935, "N", transform=ax.transAxes, ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_title("a", loc="left", fontweight="bold"); ax.set_axis_off()

# (b, c) exemplar trajectories: one recovered, one censored at harvest (mid-season onset, to avoid pre-harvest defoliation)
ex_pool = E[(E.crop == "cotton") & (E.acute == 1)]
ARW = dict(arrowstyle="-|>", color="black", lw=1.2, shrinkA=2, shrinkB=4)
TXT = dict(ha="center", va="center", fontsize=10, fontweight="bold", color="black")
for k, (rec, ttl) in enumerate([(1, "b  an episode that taps (recovers)"), (0, "c  an episode that persists to harvest")]):
    axk = fig.add_subplot(gs_[k, 1:])
    cand = ex_pool[(ex_pool.recovered == rec) & (ex_pool.dur_d.between(36, 84))]
    if rec == 0 and (cand.onset_stage == "mid-season").any(): cand = cand[cand.onset_stage == "mid-season"]
    if len(cand):
        e = cand.sample(1, random_state=PARAMS["seed"] + k).iloc[0]; tr = D[(D.parcel_id == e.parcel_id) & (D.year == e.year)]
        axk.plot(cidx_to_doy(tr.cidx), tr.z_evi2, "-o", ms=3, color="#0fe36e", label="EVI2 anomaly")
        axk.plot(cidx_to_doy(tr.cidx), tr.z_ndmi, "-", color="#1e18e0", lw=1, label="NDMI anomaly")
        axk.axhline(PARAMS["z_stress_evi2"], color="#ef3b12", lw=.8, ls="--", label=f"stress threshold (z = {PARAMS['z_stress_evi2']:g})")
        axk.axhline(PARAMS["z_recovery"], color="#3b7a57", lw=.8, ls=":", label=f"recovery threshold (z = {PARAMS['z_recovery']:g})")
        x0, x1 = cidx_to_doy(e.onset_cidx), cidx_to_doy(e.end_cidx)
        axk.axvspan(x0, x1, color="#0aebd5", alpha=.25, label="episode")
        z0 = tr.loc[tr.cidx == e.onset_cidx, "z_evi2"]; z0 = float(z0.iloc[0]) if len(z0) else PARAMS["z_stress_evi2"]
        z1 = tr.loc[tr.cidx == e.end_cidx, "z_evi2"]; z1 = float(z1.iloc[0]) if len(z1) else PARAMS["z_recovery"]
        axk.annotate("Tip (onset)", xy=(x0, z0), xytext=(x0 - 28, 2.3), arrowprops=ARW, **TXT)
        axk.annotate("Tap (recovery)" if rec else "harvest (censored)", xy=(x1, z1), xytext=(x1 + 28 if rec else x1 + 18, 2.3), arrowprops=ARW, **TXT)
        print(ttl, e.parcel_id, e.year, e.crop, e.onset_stage, e.mechanism, f"onset doy {x0:.0f}, end doy {x1:.0f}")
        axk.set_title(ttl[0], loc="left", fontweight="bold"); axk.set_ylabel("anomaly (z-score)"); axk.set_ylim(-4, 3)
        if k == 1:
            axk.set_xlabel("day of year")
            axk.legend(frameon=False, fontsize=7, loc="lower left", ncol=2)
fig.tight_layout(); fig.savefig(OUT / "figures" / "fig1_system.png", bbox_inches="tight"); plt.show()

# --- GIS deliverable ---------------------------------------------------------------------------------
gis = Pm.merge(X3[["parcel_id", "exposure", "deficit_days", "nonrec_share", "exit", "orchard_conv"]], on="parcel_id", how="left")
for c in gis.columns:
    if str(gis[c].dtype) == "category": gis[c] = gis[c].astype(str)
gis.to_file(OUT / "gis" / "parcels_recovery.gpkg", layer="parcels", driver="GPKG")
E.to_parquet(OUT / "interim" / f"episodes_{tag}_final.parquet", index=False)

# --- manifest & results summary ----------------------------------------------------------------------
MANIFEST.update(dict(run_finished=time.strftime("%Y-%m-%dT%H:%M:%S"), n_parcels=int(len(PARCELS)), n_cultivated_parcel_years=int(n_py), n_episodes=int(len(E)),
                     recovered_share=float(E.recovered.mean()), acute_share=float(E.acute.mean()), mechanism_mix=E.mechanism.value_counts(normalize=True).round(3).to_dict(),
                     P_rec_36d_overall=float(1 - KaplanMeierFitter().fit(E.t, E.recovered).predict(36)), exit_rate_test=float(X3.exit.mean()), orchard_conv_rate_test=float(X3.orchard_conv.mean())))
def _json_safe(o):
    if isinstance(o, dict): return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_json_safe(v) for v in o]
    if isinstance(o, float) and not np.isfinite(o): return str(o)
    return o
(OUT / "run_manifest.json").write_text(json.dumps(_json_safe(MANIFEST), indent=1, default=str), encoding="utf-8")
hr = HR.copy()
def hr_line(v, lab):
    return f"{lab}: HR {hr.loc[v,'HR']:.2f} (95% CI {hr.loc[v,'lo']:.2f}–{hr.loc[v,'hi']:.2f}, p={hr.loc[v,'p']:.3f})" if v in hr.index else f"{lab}: n/a"
(OUT / "RESULTS_SUMMARY.md").write_text(summary, encoding="utf-8"); print(summary)
print("outputs:", sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file() and "interim" not in str(p)))

## 1.17 · Exports for the ArcGIS Pro spatial analysis

Parcel layer with mulch share for MGWR, and the parcel × half-season table for the emerging hot-spot analysis (14 half-season steps).

In [ ]:
# parcel-level mulch share (share of seasons flagged mulched) — needed by MGWR as an explanatory variable
PARCELS["mulch_share"] = PARCELS.parcel_id.map(PMLI.groupby("parcel_id").mulch.mean()).fillna(0)
gis2 = gis.merge(PARCELS[["parcel_id", "mulch_share"]], on="parcel_id", how="left")
gis2["log_area"] = np.log(gis2.area_ha); gis2["log_ntl"] = np.log1p(gis2.ntl.clip(lower=0))
gis2.to_file(OUT / "gis" / "parcels_recovery.gpkg", layer="parcels", driver="GPKG")

# parcel-season table for the space-time cube: one row per cultivated parcel-season, unrecovered stress-days and episode count
ps = E.groupby(["parcel_id", "year"]).agg(nonrec_days=("nonrec_days", "sum"), episodes=("dur_d", "size")).reset_index()
ps = PY[PY.cultivated == 1][["parcel_id", "year"]].merge(ps, on=["parcel_id", "year"], how="left").fillna({"nonrec_days": 0, "episodes": 0})
cent = PARCELS.set_index("parcel_id").geometry.centroid
ps["lon"] = ps.parcel_id.map(cent.x); ps["lat"] = ps.parcel_id.map(cent.y)
ps["date"] = pd.to_datetime(ps.year.astype(str) + "-07-15")          # one time slice per season
ps.to_csv(OUT / "gis" / "parcel_season_nonrec.csv", index=False)
print(f"ArcGIS inputs written: parcels_recovery.gpkg ({len(gis2):,} parcels, +mulch_share) and parcel_season_nonrec.csv ({len(ps):,} parcel-seasons)")


In [ ]:
# parcel × half-season table for the emerging hot-spot analysis (arcpy_stepC_halfseason.py): each season split at composite 12 (~23 July)
# by episode onset, dated 1 March / 1 September (14 six-month steps); rel_nonrec = non-recovered stress-days / oasis-wide mean of the same half-season
E["half"] = np.where(E.onset_cidx < 12, "early", "late")
hs = E.groupby(["parcel_id", "year", "half"]).agg(nonrec_days=("nonrec_days", "sum"), episodes=("dur_d", "size")).reset_index()
hs = PY[PY.cultivated == 1][["parcel_id", "year"]].merge(pd.DataFrame({"half": ["early", "late"]}), how="cross").merge(hs, on=["parcel_id", "year", "half"], how="left").fillna({"nonrec_days": 0, "episodes": 0})
hs["rel_nonrec"] = hs.nonrec_days / hs.groupby(["year", "half"]).nonrec_days.transform("mean")
hs["lon"] = hs.parcel_id.map(cent.x); hs["lat"] = hs.parcel_id.map(cent.y)
hs["date"] = pd.to_datetime(hs.year.astype(str) + np.where(hs.half == "early", "-03-01", "-09-01"))
hs.to_csv(OUT / "gis" / "parcel_halfseason_nonrec.csv", index=False)
print(f"parcel_halfseason_nonrec.csv ({len(hs):,} parcel-half-seasons)")

## 1.18 · Targeting at canal-command scale (Fig. 9)

Rehabilitation budgets are spent on canals and their command areas, not on individual parcels. Using the intake each parcel is served by (tier 1 and tier 2 of the directed network), parcels are grouped into **canal commands** and the H4 test is repeated at that level: commands are ranked on their 2019–2022 record and the share of 2023–2025 unrecovered stress-days captured is plotted against the share of commands (and of command area) treated. Because a command's outcome is the sum over its parcels, the curves are steeper than the parcel-level ones by construction; the comparison that matters is between rules, at the same budget.

In [ ]:
if HAS_HYD and "intake_id" in PARCELS.columns and PARCELS.intake_id.notna().any():
    TC = TG.merge(PARCELS[["parcel_id", "intake_id", "hyd_tier"]], on="parcel_id", how="left")
    TC = TC[TC.intake_id.notna() & (TC.hyd_tier <= 2)]
    CMD = TC.groupby("intake_id").agg(n_parcels=("parcel_id", "size"), area_ha=("area_ha", "sum"), exposure=("exposure", "mean"), deficit_days=("deficit_days", "mean"),
                                       rci=("rci", "mean"), nonrec_days_test=("nonrec_days_test", "sum")).reset_index()
    CMD = CMD[CMD.n_parcels >= 10].copy(); CMD["rci_deficit"] = 1 - CMD.rci; CMD["random"] = np.random.default_rng(PARAMS["seed"]).random(len(CMD)); CMD["small_first"] = -CMD.area_ha / CMD.n_parcels
    rows = []
    for lab, sc in RULES.items():
        rows.append(dict(rule=lab, capture_10pct=capture_at(CMD, sc, .10), capture_20pct=capture_at(CMD, sc, .20), capture_30pct=capture_at(CMD, sc, .30)))
    T9 = pd.DataFrame(rows).set_index("rule")
    T9.loc["random"] = [0.10, 0.20, 0.30]                                       # expected capture under random ordering = budget share
    T9.to_csv(OUT / "tables" / "table9_targeting_by_command.csv"); display(T9.round(3))
    MANIFEST["H4_targeting_by_command"] = dict(n_commands=int(len(CMD)), capture_20pct=T9.capture_20pct.round(3).to_dict())
    RULE_COLORS = ["#00B050", "#FF0000", "#0070C0", "#ED7D31", "#7F7F7F"]       # exposure, deficit, RCI, smallest-first, random
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    for (lab, sc), c in zip(RULES.items(), RULE_COLORS):
        ls = "--" if sc == "random" else "-"
        if sc == "random":
            R = np.random.default_rng(PARAMS["seed"])
            grid = np.linspace(0, 1, 201)
            ys0, ys1 = [], []
            for _ in range(500):
                CMD["_r"] = R.random(len(CMD))
                x, y = gain(CMD, "_r"); ys0.append(np.interp(grid, x, y))
                x, y = gain(CMD, "_r", weight="area_ha"); ys1.append(np.interp(grid, x, y))
            CMD = CMD.drop(columns="_r")
            axes[0].plot(grid, np.mean(ys0, 0), lw=2, color=c, ls=ls, label="random (mean of 500)")
            axes[1].plot(grid, np.mean(ys1, 0), lw=2, color=c, ls=ls, label="random (mean of 500)")
            continue
        x, y = gain(CMD, sc); axes[0].plot(x, y, lw=2, color=c, ls=ls, label=lab)
        x, y = gain(CMD, sc, weight="area_ha"); axes[1].plot(x, y, lw=2, color=c, ls=ls, label=lab)
    for ax, xl in zip(axes, [f"share of canal commands treated (n={len(CMD)})", "share of command area treated"]):
        ax.plot([0, 1], [0, 1], color="black", ls=":", lw=.8); ax.set_xlabel(xl); ax.set_ylabel("share of 2023–25 non-recovered stress-days captured"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    axes[0].legend(frameon=False, fontsize=8, loc="lower right"); axes[0].set_title("a", loc="left", fontweight="bold"); axes[1].set_title("b", loc="left", fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT / "figures" / "fig6_targeting_by_command.png", bbox_inches="tight"); plt.show()
else:
    print("command-level targeting skipped: directed hydraulic position (intake_id) not available")

## 1.19 · ArcGIS Pro results: MGWR local effects and emerging hot spots (Figs. 10–11)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Patch
SP = DATA_ROOT / "TipTap" / "hydraulic" / "spatial"
ID_NAMES = ("grid_id", "hex_id", "source_id", "objectid", "oid", "fid", "location")
MASK_NS = True                                                       # drop drivers with no significant local coefficients (SIG_PCT = 0) from the figure
MGWR_LABELS = {"INTRCPT": "Intercept", "HYD_RANK": "Head–tail rank", "HYD_DZ_M": "Fall from intake", "DIST_DESERT_KM": "Distance from oasis fringe",
               "LOG_AREA": "Log parcel area", "MULCH_SHARE": "Mulch share", "SI_PRESEASON": "Pre-season salinity index", "LOG_NTL": "Log night-time light",
               "DIST_SETTLEMENT_KM": "Settlement distance"}
def _key(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
def best_join(left, right):
    """pick the ID column pair (left, right) with the highest share of matching keys"""
    lc = [c for c in left.columns if c.lower() in ID_NAMES]
    rc = [c for c in right.columns if c.lower() in ID_NAMES]
    best = max(((a, b, _key(left[a]).isin(set(_key(right[b]))).mean()) for a in lc for b in rc), key=lambda t: t[2])
    print(f"join {best[0]} ↔ {best[1]}: {best[2]:.1%} of hexagons matched")
    if best[2] < 0.5: print("WARNING: fewer than half of the hexagons matched — check the ID fields in the ArcGIS exports")
    return best[0], best[1]
def map_frame(ax, pref, pad=0.2):
    xmin, ymin, xmax, ymax = pref.total_bounds
    ax.set_xlim(xmin - pad, xmax + pad); ax.set_ylim(ymin - pad, ymax + pad); ax.set_aspect(1 / np.cos(np.deg2rad((ymin + ymax) / 2)))
    xt = np.arange(np.ceil(xmin / 2) * 2, xmax + 1e-9, 2); yt = np.arange(np.ceil(ymin / 2) * 2, ymax + 1e-9, 2)
    ax.set_xticks(xt); ax.set_yticks(yt)
    return xmin, ymin, xmax, ymax, xt, yt
def scale_bar(ax, xmin, ymin, ymax, km=100, fs=7.5):
    L = km / (111.32 * np.cos(np.deg2rad((ymin + ymax) / 2))); x0s, y0s = xmin + 0.05, ymin + 0.1
    ax.plot([x0s, x0s + L], [y0s, y0s], color="black", lw=2.2, solid_capstyle="butt"); ax.plot([x0s, x0s + L / 2], [y0s, y0s], color="white", lw=1.2, solid_capstyle="butt")
    ax.text(x0s + L / 2, y0s + 0.1, f"{km} km", ha="center", va="bottom", fontsize=fs)

if (SP / "hex_mgwr.csv").exists() and (SP / "hex_grid.gpkg").exists():
    hexg = gpd.read_file(SP / "hex_grid.gpkg").to_crs("EPSG:4326")
    mg = pd.read_csv(SP / "hex_mgwr.csv")
    hid, mid = best_join(hexg, mg)
    hexg["_k"] = _key(hexg[hid]); mg["_k"] = _key(mg[mid])
    coef_cols = [c for c in mg.columns if re.match(r"^C\d*_", c)]                       # MGWR local coefficient fields
    pref_ll = gpd.read_file(PATHS["prefecture"]).to_crs("EPSG:4326")
    hexm = hexg.merge(mg.drop(columns=[c for c in mg.columns if c in hexg.columns and c != "_k"]), on="_k", how="left")
    bw = pd.read_csv(SP / "mgwr_bandwidths.csv") if (SP / "mgwr_bandwidths.csv").exists() else None
    if bw is not None: print("MGWR bandwidths (neighbours) per driver:"); display(bw)
    sig = dict(zip(bw.VAR_NAME.str.upper(), bw.SIG_PCT)) if bw is not None and {"VAR_NAME", "SIG_PCT"} <= set(bw.columns) else {}
    r2c = [c for c in mg.columns if c.upper().startswith("LOCALR2")]
    _v = lambda c: re.sub(r"^C\d*_", "", c).upper()
    NS = [c for c in coef_cols if sig.get(_v(c), 100) == 0] if MASK_NS else []
    plot_cols = [c for c in coef_cols if c not in NS]
    print("not significant in any hexagon (state in caption):", [MGWR_LABELS.get(_v(c), _v(c)) for c in NS])

    n = len(plot_cols); ncol = 3; nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.1 * ncol, 4.0 * nrow)); axes = np.atleast_1d(axes).ravel()
    for i, (ax, lab, c) in enumerate(zip(axes, "abcdefghijkl", plot_cols)):
        vname = _v(c)
        cax = make_axes_locatable(ax).append_axes("right", size="5%", pad=0.08)
        lim = float(np.nanpercentile(np.abs(hexm[c]), 95)) or 1
        hexm.plot(ax=ax, column=c, cmap="RdYlGn_r", vmin=-lim, vmax=lim, legend=True, cax=cax, edgecolor="none")
        cax.tick_params(labelsize=7.5)
        pref_ll.boundary.plot(ax=ax, color="black", lw=1.5)
        xmin, ymin, xmax, ymax, xt, yt = map_frame(ax, pref_ll)
        ax.set_xticklabels([f"{x:.0f}°E" for x in xt] if i // ncol == nrow - 1 or i + ncol >= n else [], fontsize=7.5)
        ax.set_yticklabels([f"{y:.0f}°N" for y in yt] if i % ncol == 0 else [], fontsize=7.5)
        ax.set_title(lab, loc="left", fontweight="bold")
    for ax in axes[n:]: ax.set_axis_off()
    # one scale bar (bottom-left panel) and one north arrow (top-right panel)
    scale_bar(axes[(nrow - 1) * ncol], xmin, ymin, ymax)
    axn = axes[ncol - 1]
    axn.annotate("", xy=(1.22, 1.20), xytext=(1.22, 1.02), xycoords="axes fraction", annotation_clip=False, arrowprops=dict(arrowstyle="-|>", lw=1.8, color="black", mutation_scale=14))
    axn.text(1.22, 1.22, "N", transform=axn.transAxes, ha="center", va="bottom", fontsize=11, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT / "figures" / "fig7_mgwr_local_effects.png", bbox_inches="tight"); plt.show()
    MANIFEST["mgwr"] = dict(n_hexagons=int(mg.shape[0]), coefficients=coef_cols, mapped=plot_cols, local_r2_median=float(mg[r2c[0]].median()) if r2c else None,
                            sig_pct={k: float(v) for k, v in sig.items()})
else:
    print("MGWR outputs not found in", SP)

if (SP / "hex_ehsa.csv").exists() and (SP / "hex_grid.gpkg").exists():
    if "hexg" not in dir() or "_k" not in hexg.columns:
        hexg = gpd.read_file(SP / "hex_grid.gpkg").to_crs("EPSG:4326"); pref_ll = gpd.read_file(PATHS["prefecture"]).to_crs("EPSG:4326")
    eh = pd.read_csv(SP / "hex_ehsa.csv")
    hid_e, eid = best_join(hexg.drop(columns="_k", errors="ignore"), eh)
    pat = [c for c in eh.columns if c.upper().startswith("PATTERN")][0]
    hexg["_ke"] = _key(hexg[hid_e]); eh["_ke"] = _key(eh[eid])
    hexe = hexg.merge(eh[["_ke", pat]], on="_ke", how="left")
    order = ["Persistent Hot Spot", "Intensifying Hot Spot", "Consecutive Hot Spot", "New Hot Spot", "Sporadic Hot Spot", "Oscillating Hot Spot", "Historical Hot Spot", "No Pattern Detected",
             "Historical Cold Spot", "Oscillating Cold Spot", "Sporadic Cold Spot", "New Cold Spot", "Consecutive Cold Spot", "Intensifying Cold Spot", "Persistent Cold Spot", "Diminishing Hot Spot", "Diminishing Cold Spot"]
    EHSA_COLORS = {"Persistent Hot Spot": "#8B0000", "Intensifying Hot Spot": "#C00000", "Consecutive Hot Spot": "#FF0000", "New Hot Spot": "#F04B4B",
                   "Sporadic Hot Spot": "#F98585", "Oscillating Hot Spot": "#F9AFAF", "Historical Hot Spot": "#FDD7D7", "Diminishing Hot Spot": "#FCC5B0",
                   "No Pattern Detected": "#E0E0E0",
                   "Historical Cold Spot": "#1F3FE0", "Oscillating Cold Spot": "#6BAED6", "Sporadic Cold Spot": "#74D49B", "New Cold Spot": "#3DBE6E",
                   "Consecutive Cold Spot": "#00A651", "Intensifying Cold Spot": "#00843D", "Persistent Cold Spot": "#00663A", "Diminishing Cold Spot": "#B8E3C2"}
    present = [o for o in order if o in set(hexe[pat].dropna())]
    fig, ax = plt.subplots(figsize=(8, 6))
    for cat in present:
        hexe[hexe[pat] == cat].plot(ax=ax, color=EHSA_COLORS.get(cat, "#BDBDBD"), edgecolor="none")
    pref_ll.boundary.plot(ax=ax, color="black", lw=1.5)
    xmin, ymin, xmax, ymax, xt, yt = map_frame(ax, pref_ll)
    ax.set_xticklabels([f"{x:.0f}°E" for x in xt], fontsize=8); ax.set_yticklabels([f"{y:.0f}°N" for y in yt], fontsize=8)
    ax.legend(handles=[Patch(facecolor=EHSA_COLORS.get(c, "#BDBDBD"), edgecolor="none", label=c) for c in present], title="Emerging hot-spot pattern",
              title_fontsize=8.5, fontsize=7.5, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0), alignment="left")
    scale_bar(ax, xmin, ymin, ymax, fs=8)
    ax.annotate("", xy=(0.88, 0.25), xytext=(0.88, 0.10), xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", lw=1.8, color="black", mutation_scale=14))
    ax.text(0.88, 0.27, "N", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT / "figures" / "fig8_emerging_hotspots.png", bbox_inches="tight"); plt.show()
    # hot-spot category vs parcel exit: join each parcel to its hexagon
    pc = gpd.GeoDataFrame(X3[["parcel_id", "exit", "orchard_conv", "deficit_days"]].merge(PARCELS[["parcel_id", "geometry"]], on="parcel_id"), geometry="geometry", crs="EPSG:4326")
    pc["geometry"] = pc.geometry.centroid
    pj = gpd.sjoin(pc, hexe[[pat, "geometry"]], how="inner", predicate="within")
    T10 = pj.groupby(pat).agg(parcels=("parcel_id", "size"), exit_rate=("exit", "mean"), orchard_conv_rate=("orchard_conv", "mean"), deficit_days=("deficit_days", "mean")).reindex([o for o in order if o in pj[pat].unique()])
    T10.to_csv(OUT / "tables" / "table10_exit_by_hotspot_category.csv"); print("Exit and conversion by emerging-hot-spot category:"); display(T10.round(3))
    MANIFEST["ehsa"] = T10.exit_rate.round(3).to_dict()
else:
    print("Emerging Hot Spot outputs not found in", SP)

## 1.20 · Maps: irrigation network and commands (Fig. 3), recovery deficit and land-use change (Fig. 13), targeting

In [ ]:
import matplotlib as mpl
from matplotlib.patches import Patch, FancyArrowPatch
from matplotlib.lines import Line2D
import pyogrio
PALETTE = ["#00B050", "#FF0000", "#0070C0", "#ED7D31", "#7030A0", "#7F7F7F"]
mpl.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "font.size": 7, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
                     "axes.linewidth": 0.6, "pdf.fonttype": 42, "ps.fonttype": 42, "axes.edgecolor": "black", "text.color": "black"})
def letter_panels(fig, axes, labels="abc", titles=None):
    fig.canvas.draw(); r = fig.canvas.get_renderer()
    for i, (ax, lab) in enumerate(zip(axes, labels)):
        bb = ax.get_tightbbox(r).transformed(fig.transFigure.inverted())
        t = fig.text(bb.x0, bb.y1 + 0.006, lab, fontsize=8, fontweight="bold", va="bottom", ha="left")
        if titles:
            tb = t.get_window_extent(r).transformed(fig.transFigure.inverted())
            fig.text(tb.x1 + 0.005, bb.y1 + 0.006, titles[i], fontsize=7, va="bottom", ha="left")
def north_arrow(ax, x=0.91, y=0.20, length=0.11, fs=8):
    ax.add_patch(FancyArrowPatch((x, y - length), (x, y), transform=ax.transAxes, arrowstyle="-|>", mutation_scale=13, lw=1.5, color="black", zorder=10))
    ax.text(x, y + 0.012, "N", transform=ax.transAxes, ha="center", va="bottom", fontsize=fs, fontweight="bold", zorder=10)
def map_frame(ax, pref, lw=1.3, scale=True, north=True, scale_km=100, fs=6.5):
    """map frame only: black neatline, black prefecture outline, scale bar bottom-left, north arrow bottom-right, no ticks"""
    pref.boundary.plot(ax=ax, color="black", lw=lw, zorder=5)
    xmin, ymin, xmax, ymax = pref.total_bounds; padx, pady = 0.03 * (xmax - xmin), 0.03 * (ymax - ymin)
    ax.set_xlim(xmin - padx, xmax + padx); ax.set_ylim(ymin - pady, ymax + pady); ax.set_aspect(1 / np.cos(np.deg2rad((ymin + ymax) / 2)))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_linewidth(0.8); s.set_color("black")
    if scale:
        km_per_deg = 111.32 * np.cos(np.deg2rad((ymin + ymax) / 2)); L = scale_km / km_per_deg
        x0 = ax.get_xlim()[0] + 0.06 * (ax.get_xlim()[1] - ax.get_xlim()[0]); y0 = ax.get_ylim()[0] + 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0])
        ax.plot([x0, x0 + L], [y0, y0], color="black", lw=2.2, solid_capstyle="butt"); ax.plot([x0, x0 + L / 2], [y0, y0], color="white", lw=1.2, solid_capstyle="butt")
        ax.text(x0 + L / 2, y0 + 0.012 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{scale_km} km", ha="center", va="bottom", fontsize=fs)
    if north: north_arrow(ax)
def legend_cell(fig, cell, handles, ncol=2, title=None):
    lax = fig.add_subplot(cell); lax.set_axis_off()
    lax.legend(handles=handles, loc="upper center", ncol=ncol, frameon=False, title=title, title_fontsize=6.5, handlelength=1.6, columnspacing=1.2, borderaxespad=0)
    return lax
def cbar_cell(fig, cell, cmap, vmin, vmax, label, extend="neither"):
    lax = fig.add_subplot(cell); lax.set_axis_off()
    cax = lax.inset_axes([0.10, 0.66, 0.80, 0.14])
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin, vmax), cmap=cmap)
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal", extend=extend); cb.set_label(label, fontsize=6.5); cb.ax.tick_params(labelsize=6, length=2)
    return lax
def footprint(pts, buf_deg=0.012):
    """compact footprint of a canal command: union of ~1.3 km buffers around its parcel centroids"""
    return gpd.GeoSeries(pts).buffer(buf_deg).union_all().simplify(0.002)

pref_ll = gpd.read_file(PATHS["prefecture"]).to_crs("EPSG:4326")
SP = DATA_ROOT / "TipTap" / "hydraulic" / "spatial"
GDBP = DATA_ROOT / "TipTap" / "hydraulic" / "hydraulic.gdb"
gdb_layers = [l[0] for l in pyogrio.list_layers(str(GDBP))] if GDBP.exists() else []
def read_gdb(name):
    return gpd.read_file(str(GDBP), layer=name).to_crs("EPSG:4326") if name in gdb_layers else None
CENT = PARCELS[["parcel_id", "geometry"]].copy(); CENT["geometry"] = CENT.geometry.centroid
OASIS = CENT.union_all().convex_hull.buffer(0.08)                                   # channels farther than ~8 km from cropland are not drawn
if (SP / "hex_grid.gpkg").exists():
    HEX = gpd.read_file(SP / "hex_grid.gpkg").to_crs("EPSG:4326")[["geometry"]].reset_index(drop=True)
else:
    from shapely.geometry import box
    xmin, ymin, xmax, ymax = pref_ll.total_bounds; dx = 2 / (111.32 * np.cos(np.deg2rad(39))); dy = 2 / 111.32
    HEX = gpd.GeoDataFrame(geometry=[box(x, y, x + dx, y + dy) for x in np.arange(xmin, xmax, dx) for y in np.arange(ymin, ymax, dy)], crs="EPSG:4326")
HEX["hex_i"] = np.arange(len(HEX))
def to_hex(df, cols, agg="mean", min_n=3):
    j = gpd.sjoin(df, HEX[["hex_i", "geometry"]], how="inner", predicate="within")
    a = j.groupby("hex_i").agg(n=("parcel_id", "size"), **{c: (c, agg) for c in cols})
    return HEX.merge(a, on="hex_i", how="inner").query("n >= @min_n")
RED = mpl.colors.LinearSegmentedColormap.from_list("red", ["#ffffff", "#FF0000"])
GRN = mpl.colors.LinearSegmentedColormap.from_list("grn", ["#ffffff", "#00B050"])
BLU = mpl.colors.LinearSegmentedColormap.from_list("blu", ["#ffffff", "#0070C0"])
PUR = mpl.colors.LinearSegmentedColormap.from_list("pur", ["#ffffff", "#7030A0"])

# ============================================================================================================
# (A) Directed irrigation network, intakes, canal commands and head-to-tail rank  -> Fig. 9
# ============================================================================================================
edges = read_gdb("water_edges")
if edges is None:
    edges = gpd.read_file(PATHS["osm"], columns=["fclass"]); edges = edges[edges.fclass.isin(["river", "canal", "drain", "stream"])].to_crs("EPSG:4326")
edges["fclass"] = edges.fclass.str.lower(); edges = gpd.clip(edges, OASIS)
P1 = PARCELS[PARCELS.hyd_tier.le(2) & PARCELS.intake_id.notna()].merge(CENT.rename(columns={"geometry": "cpt"}), on="parcel_id")
cmd_ok = P1.groupby("intake_id").size(); cmd_ok = cmd_ok[cmd_ok >= 10].index
CMDF = gpd.GeoDataFrame({"intake_id": cmd_ok.astype(float), "n_parcels": [int((P1.intake_id == i).sum()) for i in cmd_ok]},
                        geometry=[footprint(P1.loc[P1.intake_id == i, "cpt"]) for i in cmd_ok], crs="EPSG:4326")
heads = P1[P1.hyd_rank_cmd <= 0.05].groupby("intake_id").apply(lambda g: gpd.GeoSeries(g.cpt).union_all().centroid)
intk = gpd.GeoDataFrame({"intake_id": heads.index.astype(float)}, geometry=list(heads.values), crs="EPSG:4326")
lay = next((read_gdb(l) for l in ["intakes_all", "intakes_v4", "intakes"] if l in gdb_layers), None)
if lay is not None and "intake_id" in lay.columns:
    m = lay.assign(intake_id=lay.intake_id.astype(float)).set_index("intake_id").geometry
    intk["geometry"] = [m[i] if i in m.index else g for i, g in zip(intk.intake_id, intk.geometry)]
itype = PARCELS.dropna(subset=["intake_id"]).groupby("intake_id").intake_type.first() if "intake_type" in PARCELS.columns else None
intk["intake_type"] = intk.intake_id.map(itype).fillna("river").astype(str) if itype is not None else "river"
riv = intk[intk.intake_type.str.contains("river", case=False)]; pse = intk[~intk.index.isin(riv.index)]
T1p = CENT.merge(PARCELS[["parcel_id", "hyd_tier", "hyd_rank"]], on="parcel_id")
HT_CMAP = mpl.colors.LinearSegmentedColormap.from_list("ht", ["#0070C0", "#FF0000"])
sh = PARCELS.hyd_tier.value_counts(normalize=True)

fig = plt.figure(figsize=(7.2, 5.6)); gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.34], hspace=0.03, wspace=0.06)
ax = fig.add_subplot(gs[0, 0])
edges[edges.fclass.isin(["drain", "stream"])].plot(ax=ax, color="#b0b0b0", lw=0.3, zorder=2)
edges[edges.fclass == "canal"].plot(ax=ax, color=PALETTE[2], lw=0.5, zorder=3)
edges[edges.fclass == "river"].plot(ax=ax, color=PALETTE[2], lw=1.4, zorder=4)
CMDF.boundary.plot(ax=ax, color="black", lw=0.5, zorder=5)
riv.plot(ax=ax, color=PALETTE[1], marker="^", markersize=15, edgecolor="black", linewidth=0.3, zorder=6)
pse.plot(ax=ax, color=PALETTE[3], marker="^", markersize=12, edgecolor="black", linewidth=0.3, zorder=6)
map_frame(ax, pref_ll, north=False, scale=True)
legend_cell(fig, gs[1, 0], [Line2D([], [], color=PALETTE[2], lw=1.4, label="river"), Line2D([], [], color=PALETTE[2], lw=0.7, label="canal"),
                            Line2D([], [], color="#b0b0b0", lw=0.7, label="drain or stream"),
                            Line2D([], [], marker="^", color=PALETTE[1], mec="black", mew=0.3, ls="", ms=5, label=f"river intake (n = {len(riv)})"),
                            Line2D([], [], marker="^", color=PALETTE[3], mec="black", mew=0.3, ls="", ms=4.5, label=f"pseudo-intake at component head (n = {len(pse)})"),
                            Line2D([], [], color="black", lw=0.7, label=f"canal command footprint, ≥ 10 parcels (n = {len(CMDF)})")], ncol=2)
ax2 = fig.add_subplot(gs[0, 1])
T1p[T1p.hyd_tier == 3].plot(ax=ax2, color="#d9d9d9", markersize=0.8, linewidth=0, zorder=2)
T1p[T1p.hyd_tier == 2].plot(ax=ax2, color="#8c8c8c", markersize=0.8, linewidth=0, zorder=3)
T1p[T1p.hyd_tier == 1].plot(ax=ax2, column="hyd_rank", cmap=HT_CMAP, vmin=0, vmax=1, markersize=1.0, linewidth=0, zorder=4)
CMDF.boundary.plot(ax=ax2, color="black", lw=0.35, zorder=5)
map_frame(ax2, pref_ll, scale=False, north=True)
lax = cbar_cell(fig, gs[1, 1], HT_CMAP, 0, 1, f"head-to-tail rank within canal command (tier 1, {sh.get(1, 0):.0%} of parcels)")
lax.legend(handles=[Line2D([], [], marker="o", color="#8c8c8c", ls="", ms=3.5, label=f"tier 2: reached only undirected ({sh.get(2, 0):.0%})"),
                    Line2D([], [], marker="o", color="#d9d9d9", ls="", ms=3.5, label=f"tier 3: no mapped channel within 3 km ({sh.get(3, 0):.0%})")],
           loc="lower center", ncol=2, frameon=False, borderaxespad=0, columnspacing=1.0)
letter_panels(fig, [ax, ax2], "ab")
fig.savefig(OUT / "figures" / "fig9_network_commands.png", bbox_inches="tight"); plt.show()

# ============================================================================================================
# (B) Observed recovery deficit 2019–22, burden 2023–25, exit and orchard conversion  -> Fig. 10
# ============================================================================================================
PX = CENT.merge(TG[["parcel_id", "deficit_days", "nonrec_days_test"]], on="parcel_id").merge(X3[["parcel_id", "exit", "orchard_conv"]], on="parcel_id")
H = to_hex(PX, ["deficit_days", "nonrec_days_test", "exit", "orchard_conv"], min_n=20)
n_exit, n_conv = int(PX.exit.sum()), int(PX.orchard_conv.sum())
SEQ = "RdYlGn_r"                                                                   # green = low, yellow = mid, red = high
panels = [("deficit_days", SEQ, "non-recovered stress-days per cultivated season, 2019–22", "Recovery deficit 2019–22"),
          ("nonrec_days_test", SEQ, "non-recovered stress-days per parcel, 2023–25", "Unrecovered burden 2023–25"),
          ("exit", SEQ, f"share of parcels leaving cultivation, 2023–25 (n = {n_exit:,})", "Exit from cultivation"),
          ("orchard_conv", SEQ, f"share of annual parcels converted to orchard, 2019→2024 (n = {n_conv:,})", "Orchard conversion")]
fig = plt.figure(figsize=(7.2, 8.6)); gs = fig.add_gridspec(4, 2, height_ratios=[1, 0.24, 1, 0.24], hspace=0.03, wspace=0.06)
axs = []
for i, (col, cmap, lab, ttl) in enumerate(panels):
    r, c = (i // 2) * 2, i % 2
    ax = fig.add_subplot(gs[r, c]); axs.append(ax)
    vmax = float(np.nanpercentile(H[col], 97)) or float(H[col].max())
    HEX.merge(H[["hex_i"]], on="hex_i").plot(ax=ax, color="#f0f0f0", edgecolor="none", zorder=1)          # cropland base
    H.plot(ax=ax, column=col, cmap=cmap, vmin=0, vmax=vmax, edgecolor="none", zorder=2)
    map_frame(ax, pref_ll, lw=1.1, scale=(i == 2), north=(i == 3))
    cbar_cell(fig, gs[r + 1, c], cmap, 0, vmax, lab, extend="max")
letter_panels(fig, axs, "abcd")
fig.savefig(OUT / "figures" / "fig10_deficit_exit_map.png", bbox_inches="tight"); plt.show()

# ============================================================================================================
# (C) Targeting map: commands at a 30% budget (categorical), parcels at a 20% budget (share selected per hexagon)  -> Fig. 11
# ============================================================================================================
def top_set(df, score, frac):
    k = int(round(frac * len(df))); return set(df.sort_values(score, ascending=False).head(k).index)
sel = {}
Tp = TG.set_index("parcel_id"); k20 = 0.20
p_rci, p_def = top_set(Tp, "rci_deficit", k20), top_set(Tp, "deficit_days", k20)
Tp["sel_rci"] = Tp.index.isin(p_rci).astype(int); Tp["sel_def"] = Tp.index.isin(p_def).astype(int)
Tp["captured"] = Tp.nonrec_days_test / Tp.nonrec_days_test.sum()
sel["parcel"] = {"structural RCI (20%)": float(Tp.loc[Tp.sel_rci == 1, "captured"].sum()), "recovery deficit (20%)": float(Tp.loc[Tp.sel_def == 1, "captured"].sum()),
                 "overlap of the two selections": float((Tp.sel_rci & Tp.sel_def).sum() / max(Tp.sel_rci.sum(), 1))}
HS = to_hex(CENT.merge(Tp[["sel_rci", "sel_def"]], left_on="parcel_id", right_index=True), ["sel_rci", "sel_def"])
SEL_COLORS = {"both": PALETTE[4], "rci": PALETTE[2], "deficit": PALETTE[0], "neither": "#e6e6e6"}
SEL_LABELS = {"both": "selected by both rules", "rci": "structural RCI only", "deficit": "recovery deficit only", "neither": "not selected"}
have_cmd = "CMD" in globals() and len(CMD) > 0
if have_cmd:
    C = CMD.set_index("intake_id"); C.index = C.index.astype(float); k30 = 0.30
    s_rci, s_def = top_set(C, "rci_deficit", k30), top_set(C, "deficit_days", k30)
    C["sel"] = np.select([C.index.isin(s_rci & s_def), C.index.isin(s_rci - s_def), C.index.isin(s_def - s_rci)], ["both", "rci", "deficit"], "neither")
    C["captured"] = C.nonrec_days_test / C.nonrec_days_test.sum()
    CMDS = CMDF.merge(C[["sel", "captured"]], left_on="intake_id", right_index=True, how="left").fillna({"sel": "neither"})
    sel["command"] = {"structural RCI (30%)": float(C.loc[C.sel.isin(["both", "rci"]), "captured"].sum()), "recovery deficit (30%)": float(C.loc[C.sel.isin(["both", "deficit"]), "captured"].sum()),
                      "commands selected by both": int((C.sel == "both").sum())}
print("targeting maps — share of the 2023–25 burden captured:", sel)

fig = plt.figure(figsize=(7.2, 4.4)); gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.55], hspace=0.03, wspace=0.06)
ax = fig.add_subplot(gs[0, 0])
if have_cmd:
    CMDS[CMDS.sel == "neither"].plot(ax=ax, color=SEL_COLORS["neither"], edgecolor="black", linewidth=0.3, zorder=2)
    CMDS[CMDS.sel != "neither"].plot(ax=ax, color=CMDS.loc[CMDS.sel != "neither", "sel"].map(SEL_COLORS), edgecolor="black", linewidth=0.3, zorder=3)
    legend_cell(fig, gs[1, 0], [Patch(facecolor=SEL_COLORS[k], edgecolor="black", lw=0.3, label=SEL_LABELS[k]) for k in ["both", "rci", "deficit", "neither"]], ncol=2,
                title=f"30% of {len(CMDS)} commands\nRCI captures {sel['command']['structural RCI (30%)']:.0%}, deficit {sel['command']['recovery deficit (30%)']:.0%}")
else:
    ax.text(0.5, 0.5, "command-level selection unavailable (run §15 first)", transform=ax.transAxes, ha="center", va="center")
map_frame(ax, pref_ll, lw=1.1, scale=True, north=False)
ax2 = fig.add_subplot(gs[0, 1]); ax3 = fig.add_subplot(gs[0, 2])
for a, col, cmap, lab in [(ax2, "sel_def", GRN, f"share of parcels selected,\nrecovery-deficit rule (captures {sel['parcel']['recovery deficit (20%)']:.0%})"),
                          (ax3, "sel_rci", BLU, f"share of parcels selected,\nstructural RCI rule (captures {sel['parcel']['structural RCI (20%)']:.0%})")]:
    HEX.merge(HS[["hex_i"]], on="hex_i").plot(ax=a, color="#f0f0f0", edgecolor="none", zorder=1)
    HS.plot(ax=a, column=col, cmap=cmap, vmin=0, vmax=1, edgecolor="none", zorder=2)
    map_frame(a, pref_ll, lw=1.1, scale=False, north=(a is ax3))
    cbar_cell(fig, gs[1, 1 if a is ax2 else 2], cmap, 0, 1, lab)
letter_panels(fig, [ax, ax2, ax3], "abc")
fig.savefig(OUT / "figures" / "fig11_targeting_map.png", bbox_inches="tight"); plt.show()
MANIFEST["targeting_maps"] = sel

# Part 2 · Parcel sample and fixed boundaries (reviewer comments 10–11)

## 2.1 · Setup and inputs

In [ ]:
from __future__ import annotations
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely import wkb
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "font.size": 8, "axes.spines.top": False, "axes.spines.right": False})

ROOT = Path(os.environ.get("TIPTAP_ROOT", r"D:\BRA\TipTap"))
OUT  = Path(os.environ.get("TIPTAP_R1011_OUT", str(ROOT / "outputs_recovery" / "revision" / "R10_R11")))
for sub in ["tables", "figures"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

P = dict(
    meta      = {y: ROOT / "outputs_recovery" / "interim" / f"parcel_meta_full_{y}.parquet" for y in range(2019, 2026)},
    struct    = ROOT / "outputs_recovery" / "interim" / "parcels_struct_full.parquet",
    seasons   = ROOT / "outputs_recovery" / "gis" / "parcel_season_nonrec.csv",
    episodes  = ROOT / "outputs_recovery" / "interim" / "episodes_full_final.parquet",
    hydraulic = ROOT / "hydraulic" / "hydraulic_position.csv",
    hetero    = ROOT / "outputs_recovery" / "revision" / "R11_parcel_heterogeneity.csv",   # optional
)
for k, v in P.items():
    if isinstance(v, dict):
        for y, f in v.items(): assert f.exists(), f"missing {f}"
    elif k != "hetero":
        assert v.exists(), f"missing {v}"
print("root:", ROOT, "\nout :", OUT, "\nheterogeneity file present:", P["hetero"].exists())

# Frozen parameters copied from the pipeline (TipTap_02_recovery_inequality_pipeline_v3.ipynb, §1)
PARAMS = dict(seed=20260913, years=list(range(2019, 2026)), baseline_years=[2019, 2020, 2021, 2022], test_years=[2023, 2024, 2025],
              min_pure_px_10=6, fallow_evi2_max=0.25, size_bins_ha=[0, 0.5, 2, 10, np.inf], size_labels=["<0.5 ha", "0.5–2 ha", "2–10 ha", ">10 ha"],
              export_chunk=4000)
np.random.seed(PARAMS["seed"])

# --- Upstream counts that are NOT reproducible from this folder. Fill in from the GEE export script / Wu et al. (2025) layer.
UPSTREAM = dict(
    wu2025_parcels_kashgar      = None,   # parcels in the Wu et al. (2025) layer intersecting Kashgar Prefecture
    wu2025_area_ha_kashgar      = None,   # their total area (ha)
    gee_interior_pixel_rule     = None,   # interior 10 m pixel threshold applied in the GEE export (pipeline §2 text says ≥ 22; verify)
    gee_exported_parcels        = None,   # parcels that passed the GEE filter and were exported (before chunk losses)
)

## 2.2 · Source parcel layer and export coverage

In [ ]:
# source layer: Wu et al. (2025) parcels intersecting Kashgar (fragments dissolved by FID_Kashga; -1 = no source parcel)
import geopandas as gpd
wu = gpd.read_file(ROOT.parent / "Land parcel" / "kashgar_parcels.shp", ignore_geometry=True)
wu = wu[wu.FID_Kashga != -1]
UPSTREAM.update(wu2025_parcels_kashgar=int(wu.FID_Kashga.nunique()), wu2025_area_ha_kashgar=round(float(wu.groupby("FID_Kashga").Area.first().sum())),
                gee_interior_pixel_rule="one-pixel inward buffer",
                gee_exported_parcels=int(max(pd.read_parquet(f, columns=["parcel_id"]).parcel_id.max() for f in P["meta"].values())) + 1)   # export ids run 0..n-1
print(UPSTREAM)

In [ ]:
meta = {}
for y, f in P["meta"].items():
    m = pd.read_parquet(f)
    m["year"] = y
    meta[y] = m
cov = {y: set(m.parcel_id) for y, m in meta.items()}
union = sorted(set().union(*cov.values()))
balanced = set.intersection(*cov.values())
print(f"parcels in any yearly export: {len(union):,} | in all seven exports: {len(balanced):,}")

rows = []
for y, s in cov.items():
    miss = np.array(sorted(set(union) - s))
    if len(miss):
        brk = np.where(np.diff(miss) > 1)[0]
        runs = [(int(a), int(b), int(b - a + 1)) for a, b in zip(np.r_[miss[0], miss[brk + 1]], np.r_[miss[brk], miss[-1]])]
    else:
        runs = []
    rows.append(dict(year=y, parcels_exported=len(s), parcels_missing=len(miss), share_of_union=len(s) / len(union),
                     missing_id_runs=str(runs)))
COVER = pd.DataFrame(rows)
display(COVER)

# Are the gaps whole export chunks (multiples of the 4,000-parcel chunk used by the GEE export) rather than parcel-level exclusions?
chunk = PARAMS["export_chunk"]
for r in rows:
    runs = eval(r["missing_id_runs"])
    whole = [ (a % chunk == 0) and (n == chunk) for a, b, n in runs ]
    if runs: print(f"{r['year']}: {len(runs)} missing runs; whole {chunk}-parcel chunks: {sum(whole)} of {len(runs)}")

## 2.3 · Geometry checks, interior-pixel rule and retained sample

In [ ]:
t0 = time.time()
U = pd.concat(meta.values()).drop_duplicates("parcel_id").sort_values("parcel_id").reset_index(drop=True)
geoms = [shape(json.loads(s)) for s in U[".geo"]]
U["geom_type_raw"] = [g.geom_type for g in geoms]
U["valid_raw"]     = [g.is_valid for g in geoms]
rep = [g.buffer(0) for g in geoms]
U["geom_type_rep"] = [g.geom_type for g in rep]
U["kept_geom"]     = U.geom_type_rep.isin(["Polygon", "MultiPolygon"])
U["kept_px"]       = U.pure_px_10 >= PARAMS["min_pure_px_10"]
print(f"geometry checks on {len(U):,} parcels in {time.time()-t0:.0f}s")
print("raw geometry types:", U.geom_type_raw.value_counts().to_dict())
print("invalid before repair:", int((~U.valid_raw).sum()), "| after buffer(0) types:", U.geom_type_rep.value_counts().to_dict())
print(f"pure_px_10: min {U.pure_px_10.min()}, parcels < 6 px: {int((~U.kept_px).sum())}, parcels < 22 px: {int((U.pure_px_10 < 22).sum())}")
print(f"area_ha: min {U.area_ha.min():.3f}, median {U.area_ha.median():.2f}, max {U.area_ha.max():.1f}")

S = pd.read_parquet(P["struct"])
retained = set(S.parcel_id)
U["retained"] = U.parcel_id.isin(retained)
assert U[U.kept_geom & U.kept_px].parcel_id.isin(retained).all() and len(retained) == int((U.kept_geom & U.kept_px).sum()), "retained set does not match the documented rules"
print(f"retained parcels (parcels_struct_full): {len(retained):,} = union {len(U):,} − geometry {int((~U.kept_geom).sum())} − pixel rule {int((U.kept_geom & ~U.kept_px).sum())}")

## 2.4 · Analytical denominators

In [ ]:
PS = pd.read_csv(P["seasons"], usecols=["parcel_id", "year", "episodes", "nonrec_days"])
E  = pd.read_parquet(P["episodes"])
assert len(E) == 266_328, len(E)

PY = pd.concat([pd.DataFrame({"parcel_id": sorted(s), "year": y}) for y, s in cov.items()], ignore_index=True)
PY = PY[PY.parcel_id.isin(retained)]
PY = PY.merge(PS[["parcel_id", "year"]].assign(cultivated=1), on=["parcel_id", "year"], how="left").fillna({"cultivated": 0})
PY["cultivated"] = PY.cultivated.astype(int)
ne = E.groupby(["parcel_id", "year"]).size().rename("n_eps").reset_index()
PY = PY.merge(ne, on=["parcel_id", "year"], how="left").fillna({"n_eps": 0})

DEN = PY.groupby("year").agg(parcel_years_with_data=("parcel_id", "size"), cultivated_parcel_seasons=("cultivated", "sum"),
                             seasons_with_episode=("n_eps", lambda s: int((s > 0).sum())), episodes=("n_eps", "sum")).reset_index()
DEN["cultivated_share"] = DEN.cultivated_parcel_seasons / DEN.parcel_years_with_data
DEN["share_seasons_with_episode"] = DEN.seasons_with_episode / DEN.cultivated_parcel_seasons
display(DEN)
tot = DEN[["parcel_years_with_data", "cultivated_parcel_seasons", "seasons_with_episode", "episodes"]].sum()
print("totals:", tot.to_dict(), "| episodes per cultivated season:", round(tot.episodes / tot.cultivated_parcel_seasons, 3))

# exit-analysis denominator exactly as in pipeline §10 (cult_base >= 3; exit = >= 1 non-cultivated test year with data)
by, ty = PARAMS["baseline_years"], PARAMS["test_years"]
pyb = PY[PY.year.isin(by)].groupby("parcel_id").cultivated.sum().rename("cult_base")
pyt = PY[PY.year.isin(ty)].groupby("parcel_id").cultivated.agg(cult_test="sum", n_test="size")
X3 = pd.concat([pyb, pyt], axis=1).dropna()
X3 = X3[X3.cult_base >= 3].copy()
X3["exit"] = ((X3.n_test - X3.cult_test) >= 1).astype(int)
print(f"exit analysis: {len(X3):,} parcels | exit rate {X3.exit.mean():.2%} | test years with data: {X3.n_test.value_counts().to_dict()}")
print("exit rate by number of test years observed:", X3.groupby('n_test').exit.mean().round(4).to_dict())

## 2.5 · Sample-flow table

In [ ]:
def ha(mask): return float(U.loc[mask, "area_ha"].sum())
fmt = lambda v: "to be supplied" if v is None else v
flow = [
    ("1. Field-parcel layer, Wu et al. (2025), parcels intersecting Kashgar Prefecture", fmt(UPSTREAM["wu2025_parcels_kashgar"]), fmt(UPSTREAM["wu2025_area_ha_kashgar"]), "source layer"),
    (f"2. Earth Engine export filter: interior 10 m pixels after one-pixel inward buffer ≥ {fmt(UPSTREAM['gee_interior_pixel_rule'])}", fmt(UPSTREAM["gee_exported_parcels"]), "to be supplied", "excludes fields narrower than ~30 m and slivers"),
    ("3. Parcels present in at least one yearly export (2019–2025)", len(U), ha(U.index >= 0), f"union of yearly exports; per-year coverage in Table S-R10b"),
    ("4. Geometry repair (buffer(0)) and polygon-type check", int(U.kept_geom.sum()), ha(U.kept_geom), f"removed {int((~U.kept_geom).sum())} GeometryCollections that were not polygons after repair"),
    ("5. Interior-pixel rule (pure_px_10 ≥ 6)", int((U.kept_geom & U.kept_px).sum()), ha(U.kept_geom & U.kept_px), f"removed {int((U.kept_geom & ~U.kept_px).sum())}; min interior pixels in the export = {int(U.pure_px_10.min())}"),
    ("6. Retained parcels (unit of analysis)", len(retained), ha(U.retained), f"median {U.loc[U.retained,'area_ha'].median():.2f} ha; size classes {S.size_class.value_counts().to_dict()}"),
    ("7. Parcel-years with satellite data", int(tot.parcel_years_with_data), np.nan, "127,245 × 7 = 890,715 possible; shortfall = missing export chunks (Table S-R10b)"),
    ("8. Cultivated parcel-seasons (seasonal max EVI2 ≥ 0.25)", int(tot.cultivated_parcel_seasons), np.nan, f"{tot.cultivated_parcel_seasons/tot.parcel_years_with_data:.1%} of parcel-years with data"),
    ("9. Parcel-seasons with ≥ 1 confirmed episode", int(tot.seasons_with_episode), np.nan, f"{tot.seasons_with_episode/tot.cultivated_parcel_seasons:.1%} of cultivated parcel-seasons"),
    ("10. Confirmed stress episodes", int(tot.episodes), np.nan, "Table 2 of the manuscript"),
    ("11. Parcels in the exit analysis (≥ 3 cultivated baseline seasons)", len(X3), np.nan, f"exit rate {X3.exit.mean():.1%}; {int((X3.n_test<3).sum()):,} parcels observed in only 2 of 3 test years"),
]
FLOW = pd.DataFrame(flow, columns=["step", "parcels_or_units", "hectares", "note"])
FLOW["hectares"] = FLOW.hectares.map(lambda v: v if isinstance(v, str) else (round(v) if pd.notna(v) else ""))
FLOW.to_csv(OUT / "tables" / "TableS_R10a_sample_flow.csv", index=False)
COVER.to_csv(OUT / "tables" / "TableS_R10b_export_coverage_by_year.csv", index=False)
DEN.to_csv(OUT / "tables" / "TableS_R10c_denominators_by_year.csv", index=False)
display(FLOW)

## 2.6 · Boundary stability: within-parcel heterogeneity flags

In [ ]:
ref = meta[2023].set_index("parcel_id")
same = {}
for y, m in meta.items():
    mm = m.set_index("parcel_id"); idx = mm.index.intersection(ref.index)
    same[y] = dict(n_common=len(idx), geo_identical=float((mm.loc[idx, ".geo"] == ref.loc[idx, ".geo"]).mean()),
                   area_identical=float(np.isclose(mm.loc[idx, "area_ha"], ref.loc[idx, "area_ha"]).mean()))
STATIC = pd.DataFrame(same).T
display(STATIC)
assert (STATIC.geo_identical == 1).all(), "geometry differs between years — the static-layer assumption does not hold as stated"
print("Confirmed: one static parcel geometry is used for all seven seasons.")

## 2.7 · Stable-parcel subsets

In [ ]:
SUBSETS = {"full sample": retained, "S1 complete coverage": balanced & retained}
HET = None
if P["hetero"].exists():
    HET = pd.read_csv(P["hetero"])
    need = {"parcel_id", "year", "evi2_peak_mean", "evi2_peak_sd"}
    assert need <= set(HET.columns), f"heterogeneity file needs columns {need}"
    HET = HET[HET.parcel_id.isin(retained) & (HET.evi2_peak_mean >= PARAMS["fallow_evi2_max"])].copy()
    HET["cv"] = HET.evi2_peak_sd / HET.evi2_peak_mean
    early = HET[HET.year <= 2021].groupby("parcel_id").cv.agg(cv_early="mean", n_early="size")
    late  = HET[HET.year >= 2023].groupby("parcel_id").cv.agg(cv_late="mean", n_late="size")
    H = pd.concat([early, late], axis=1).dropna()
    H = H[(H.n_early >= 2) & (H.n_late >= 2)]
    H["d_cv"] = H.cv_late - H.cv_early
    thr = H.d_cv.quantile(0.90)
    H["suspect"] = (H.d_cv > thr).astype(int)
    SUBSETS["S2 heterogeneity-stable"] = set(H.index[H.suspect == 0])
    print(f"heterogeneity proxy: {len(H):,} parcels with ≥2 early and ≥2 late cultivated seasons; ΔCV 90th pct = {thr:.3f}; suspect = {int(H.suspect.sum()):,}")
    H.reset_index().to_csv(OUT / "tables" / "TableS_R11a_heterogeneity_flags.csv", index=False)
else:
    print("R11_parcel_heterogeneity.csv not found — S2 pending (run extract_parcel_heterogeneity.py on the full panel)")
SUBDEF = pd.DataFrame([(k, len(v), len(v) / len(retained)) for k, v in SUBSETS.items()], columns=["subset", "parcels", "share_of_retained"])
SUBDEF.to_csv(OUT / "tables" / "TableS_R11b_subset_definition.csv", index=False); display(SUBDEF)

## 2.8 · Episode census and recovery by subset

Same quantities as Table 2 (episodes, unresolved share, Kaplan–Meier P(recover ≤ 36 d), share of cultivated parcel-seasons with ≥ 1 episode, episodes per season).

In [ ]:
def km_p(df, h=36):
    km = KaplanMeierFitter().fit(df.t, df.recovered)
    return float(1 - km.survival_function_at_times(h).values[0])
rows = []
for k, ids in SUBSETS.items():
    e = E[E.parcel_id.isin(ids)]; py = PY[PY.parcel_id.isin(ids) & (PY.cultivated == 1)]
    rows.append(dict(subset=k, parcels=len(ids), episodes=len(e), unresolved=int((e.recovered == 0).sum()), unresolved_share=float((e.recovered == 0).mean()),
                     P_rec_36d=km_p(e), share_seasons_ge1=float((py.n_eps > 0).mean()), episodes_per_season=len(e) / len(py),
                     P_rec_36d_excl_late=km_p(e[e.onset_stage != "late-season"])))
CENSUS = pd.DataFrame(rows); CENSUS.to_csv(OUT / "tables" / "TableS_R11c_census_by_subset.csv", index=False); display(CENSUS.round(3))

## 2.9 · Cox recovery model by subset

In [ ]:
NUM = ["log_area", "hyd_rank", "hyd_dz_m", "hyd_unmapped", "log_dist_desert", "mulch", "log_ntl", "dist_settlement_km", "dist_road_km", "z_onset", "demand"]
CAT = {"crop": "cotton", "onset_stage": "development", "mechanism": "water-led", "salinity_class": "mid", "year": 2021}
STRUCT = ["log_area", "hyd_rank", "hyd_dz_m", "hyd_unmapped", "log_dist_desert", "mulch", "log_ntl", "dist_settlement_km", "dist_road_km", "salinity_class_high", "salinity_class_low"]
def design(df):
    X = df[NUM].copy()
    for c, r in CAT.items():
        d = pd.get_dummies(df[c].astype(str), prefix=c, dtype=float).drop(columns=[f"{c}_{r}"], errors="ignore")
        X = pd.concat([X, d], axis=1)
    return X.fillna(0)

def fit_cox(e):
    e = e[e.crop != "unclassified"].copy(); e["cell_code"] = pd.factorize(e.cell)[0]
    X = design(e); tr = e.year.isin(PARAMS["baseline_years"]).values
    Xtr = X[tr].copy(); Xtr["t"] = e.t[tr].values; Xtr["recovered"] = e.recovered[tr].values; Xtr["cell_code"] = e.cell_code[tr].values
    Xtr = Xtr.loc[:, (Xtr.std() > 0) | Xtr.columns.isin(["t", "recovered", "cell_code"])]
    cph = CoxPHFitter(penalizer=0.01).fit(Xtr, duration_col="t", event_col="recovered", cluster_col="cell_code", robust=True)
    feats = [c for c in Xtr.columns if c not in ("t", "recovered", "cell_code")]
    c_tr = concordance_index(e.t[tr], -cph.predict_partial_hazard(Xtr[feats]), e.recovered[tr])
    c_te = concordance_index(e.t[~tr], -cph.predict_partial_hazard(X[~tr][feats]), e.recovered[~tr])
    hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].rename(columns={"exp(coef)": "HR", "exp(coef) lower 95%": "lo", "exp(coef) upper 95%": "hi"})
    return hr, dict(n_train=int(tr.sum()), n_test=int((~tr).sum()), c_train=c_tr, c_test=c_te)

COX, FITS = {}, {}
for k, ids in SUBSETS.items():
    t0 = time.time(); COX[k], FITS[k] = fit_cox(E[E.parcel_id.isin(ids)])
    print(f"{k}: {FITS[k]} ({time.time()-t0:.0f}s)")

# reproduction check against the published table
pub = pd.read_csv(ROOT / "outputs_recovery" / "tables" / "table2_cox_hazard_ratios.csv", index_col=0)
chk = pub[["HR"]].join(COX["full sample"][["HR"]], rsuffix="_refit")
print("max |HR − HR_refit| on the full sample:", float((chk.HR - chk.HR_refit).abs().max()))

In [ ]:
cmp = pd.concat({k: v.loc[[i for i in STRUCT if i in v.index]] for k, v in COX.items()}, axis=1)
cmp.columns = [f"{a} | {b}" for a, b in cmp.columns]
cmp.to_csv(OUT / "tables" / "TableS_R11d_cox_by_subset.csv")
pd.DataFrame(FITS).T.to_csv(OUT / "tables" / "TableS_R11e_cox_fit_by_subset.csv")
display(cmp.round(3))
# sign/significance agreement between full and subsets
for k in list(SUBSETS)[1:]:
    a, b = COX["full sample"].loc[STRUCT], COX[k].loc[STRUCT]
    same_sign = (np.sign(np.log(a.HR)) == np.sign(np.log(b.HR)))
    within_ci = (b.HR >= a.lo) & (b.HR <= a.hi)
    print(f"{k}: same direction {int(same_sign.sum())}/{len(STRUCT)}; subset HR inside full-sample 95% CI {int(within_ci.sum())}/{len(STRUCT)}")

## 2.10 · Exit model by subset

In [ ]:
# hydraulic covariates exactly as built in pipeline §4.7 (parcels_struct_full.parquet predates that step)
hyd = pd.read_csv(P["hydraulic"])
SH = S.drop(columns=[c for c in hyd.columns if c != "parcel_id" and c in S.columns]).merge(hyd, on="parcel_id", how="left")
SH["hyd_tier"] = SH.hyd_tier.fillna(3).astype(int); SH["hyd_unmapped"] = (SH.hyd_tier != 1).astype(int)
SH["hyd_rank"] = SH.hyd_rank_cmd.fillna(SH.hyd_rank_county)
SH["hyd_rank"] = SH.hyd_rank.fillna(SH.groupby("county").hyd_rank.transform("median")).fillna(0.5)
SH["hyd_dz_m"] = SH.hyd_dz_m.fillna(SH.groupby("county").hyd_dz_m.transform("median")).fillna(0)
_e = E.drop_duplicates("parcel_id").set_index("parcel_id"); _p = SH.set_index("parcel_id").loc[_e.index]
assert all(np.isclose(_e[c], _p[c]).all() for c in ["hyd_rank", "hyd_dz_m", "hyd_unmapped"]), "hydraulic reconstruction differs from the episode table"

Eb = E[E.year.isin(by)].groupby("parcel_id").agg(eps_base=("dur_d", "size"), nonrec_share=("recovered", lambda s: 1 - s.mean()), nonrec_days_base=("nonrec_days", "sum"))
Xe = SH.drop(columns="geometry").merge(X3, on="parcel_id").merge(Eb, on="parcel_id", how="left")
Xe["exposure"] = Xe.eps_base.fillna(0) / Xe.cult_base; Xe["deficit_days"] = Xe.nonrec_days_base.fillna(0) / Xe.cult_base; Xe["nonrec_share"] = Xe.nonrec_share.fillna(0)
Xe["cell_code"] = pd.factorize(Xe.cell)[0]
F3 = "exit ~ deficit_days + exposure + nonrec_share + log_area + log_dist_desert + hyd_rank + hyd_dz_m + hyd_unmapped + C(salinity_class, Treatment('mid')) + log_ntl + dist_settlement_km + dist_road_km + C(county)"
EXIT = {}
for k, ids in SUBSETS.items():
    d = Xe[Xe.parcel_id.isin(ids)].dropna(subset=["deficit_days", "exposure", "nonrec_share", "log_area", "log_dist_desert", "hyd_rank", "hyd_dz_m", "log_ntl", "dist_settlement_km", "dist_road_km", "salinity_class", "county"])
    m = smf.logit(F3, data=d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(d.cell_code)[0]}, maxiter=300)
    tab = pd.DataFrame({"OR": np.exp(m.params), "lo": np.exp(m.conf_int()[0]), "hi": np.exp(m.conf_int()[1]), "p": m.pvalues})
    tab = tab.loc[[i for i in tab.index if not i.startswith(("C(county", "Intercept"))]]
    EXIT[k] = tab; print(f"{k}: n={len(d):,}, exits={int(d.exit.sum()):,} ({d.exit.mean():.2%}); OR per non-recovered stress-day = {tab.loc['deficit_days','OR']:.4f} (p={tab.loc['deficit_days','p']:.1e})")
pubx = pd.read_csv(ROOT / "outputs_recovery" / "tables" / "table6_H3_exit.csv", index_col=0)
print("published OR(deficit_days):", round(float(pubx.loc["deficit_days", "OR"]), 4))
cmpx = pd.concat(EXIT, axis=1); cmpx.columns = [f"{a} | {b}" for a, b in cmpx.columns]
cmpx.to_csv(OUT / "tables" / "TableS_R11f_exit_by_subset.csv"); display(cmpx.round(3))

## 2.11 · Subset comparison figure

In [ ]:
labels = {"log_area": "log parcel area", "hyd_rank": "head→tail rank", "hyd_dz_m": "fall from intake (m)", "hyd_unmapped": "no mapped channel", "log_dist_desert": "log distance from fringe",
          "mulch": "mulched drip", "log_ntl": "log night-time light", "dist_settlement_km": "distance to settlement", "dist_road_km": "distance to road",
          "salinity_class_high": "high pre-season salinity", "salinity_class_low": "low pre-season salinity", "deficit_days": "non-recovered stress-days / season", "exposure": "episodes / season", "nonrec_share": "non-recovered share"}
cols = ["#1f3b73", "#c8553d", "#3b7a57"]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.8), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axes[0]; ks = list(SUBSETS); yy = np.arange(len(STRUCT))
for j, k in enumerate(ks):
    h = COX[k].reindex(STRUCT); off = (j - (len(ks) - 1) / 2) * 0.25
    ax.errorbar(h.HR, yy + off, xerr=[h.HR - h.lo, h.hi - h.HR], fmt="o", ms=3, color=cols[j], ecolor=cols[j], capsize=1.5, lw=.8, label=k)
ax.axvline(1, color="k", lw=.6); ax.set_xscale("log"); ax.set_yticks(yy); ax.set_yticklabels([labels[s] for s in STRUCT]); ax.invert_yaxis(); ax.set_ylim(len(STRUCT) - 0.5, -1.3)
ax.set_xlabel("hazard ratio for recovery (log scale)"); ax.set_title("a", loc="left", fontweight="bold"); ax.legend(frameon=False, fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2)
ax.set_xticks([0.85, 0.9, 1.0, 1.1, 1.2, 1.3]); ax.set_xticklabels(["0.85", "0.9", "1.0", "1.1", "1.2", "1.3"]); ax.minorticks_off()
ax = axes[1]; xs = ["deficit_days", "exposure", "nonrec_share", "log_area", "log_dist_desert", "hyd_rank", "hyd_unmapped", "log_ntl"]; yy = np.arange(len(xs))
for j, k in enumerate(ks):
    h = EXIT[k].reindex(xs); off = (j - (len(ks) - 1) / 2) * 0.25
    ax.errorbar(h.OR, yy + off, xerr=[h.OR - h.lo, h.hi - h.OR], fmt="o", ms=3, color=cols[j], ecolor=cols[j], capsize=1.5, lw=.8)
ax.axvline(1, color="k", lw=.6); ax.set_xscale("log"); ax.set_yticks(yy); ax.set_yticklabels([labels[s] for s in xs]); ax.invert_yaxis()
ax.set_xlabel("odds ratio for exit (log scale)"); ax.set_title("b", loc="left", fontweight="bold")
ax.set_xticks([0.4, 0.6, 1.0, 1.5, 2.5]); ax.set_xticklabels(["0.4", "0.6", "1.0", "1.5", "2.5"]); ax.minorticks_off()
fig.tight_layout(); fig.savefig(OUT / "figures" / "FigS_R11_stable_subset.png", bbox_inches="tight"); fig.savefig(OUT / "figures" / "FigS_R11_stable_subset.pdf", bbox_inches="tight"); plt.show()

# Part 3 · Cultivation rule and crop maps (reviewer comments 12–13)

## 3.1 · EVI2 cultivation rule against the crop maps; 2024 map reused for 2025

In [ ]:
"""R12/R13: EVI2 cultivation rule vs crop map; consequences of reusing the 2024 crop map for 2025.
Inputs: crop_majority_2019_2024.parquet (from crop_majority.py), episodes_full_final.parquet, parcel_season_nonrec.csv,
parcel_meta_full_YYYY.parquet, parcels_struct_full.parquet, hydraulic_position.csv. Read-only.
"""
import os, numpy as np, pandas as pd
from pathlib import Path
from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter
from sklearn.metrics import roc_auc_score
import statsmodels.formula.api as smf
ROOT = Path(os.environ.get("TIPTAP_ROOT", r"D:\BRA\TipTap")); UP = ROOT / "outputs_recovery"; OUT = UP / "revision" / "R12_R13"
CR = pd.read_parquet(OUT / "crop_majority_2019_2024.parquet")
E = pd.read_parquet(UP / "interim/episodes_full_final.parquet")
PS = pd.read_csv(UP / "gis/parcel_season_nonrec.csv", usecols=["parcel_id", "year"]).assign(cultivated=1)
cov = {y: set(pd.read_parquet(UP / f"interim/parcel_meta_full_{y}.parquet", columns=["parcel_id"]).parcel_id) for y in range(2019, 2026)}
PY = pd.concat([pd.DataFrame({"parcel_id": sorted(s), "year": y}) for y, s in cov.items()]).merge(PS, how="left").fillna({"cultivated": 0})
PY["cultivated"] = PY.cultivated.astype(int)

# ---------- Comment 12: EVI2 rule vs crop map (true maps 2019-2024)
X = PY.merge(CR.rename(columns={"map_year": "year"})[["parcel_id", "year", "crop", "purity"]], on=["parcel_id", "year"], how="inner")
X["map_cropped"] = (X.crop != "unclassified").astype(int)
def metrics(d):
    tp = ((d.cultivated == 1) & (d.map_cropped == 1)).sum(); fn = ((d.cultivated == 0) & (d.map_cropped == 1)).sum()
    tn = ((d.cultivated == 0) & (d.map_cropped == 0)).sum(); fp = ((d.cultivated == 1) & (d.map_cropped == 0)).sum()
    sens = tp / (tp + fn); spec = tn / (tn + fp)
    return dict(n=len(d), tp=tp, fn=fn, fp=fp, tn=tn, sensitivity=sens, specificity=spec, balanced_acc=0.5 * (sens + spec), overall_agreement=(tp + tn) / len(d),
                map_unclassified_share_when_evi2_fallow=fp / (fp + tn) if (fp + tn) else np.nan)
rows = [dict(year="all", **metrics(X))] + [dict(year=y, **metrics(g)) for y, g in X.groupby("year")]
M = pd.DataFrame(rows); print(M.round(3).to_string()); M.to_csv(OUT / "TableS_R12a_evi2_rule_vs_cropmap.csv", index=False)
# share unclassified among EVI2-fallow vs EVI2-cultivated (the manuscript's 80% vs 4%)
print("map 'unclassified' share | EVI2 fallow:", round(1 - X[X.cultivated == 0].map_cropped.mean(), 3), "| EVI2 cultivated:", round(1 - X[X.cultivated == 1].map_cropped.mean(), 3))
# by crop class: share EVI2-cultivated
bc = X.groupby("crop").cultivated.agg(["mean", "size"]); print(bc.round(3)); bc.to_csv(OUT / "TableS_R12b_evi2_cultivated_by_mapclass.csv")
# purity effect
X["pure"] = X.purity >= 0.8
print("agreement by purity>=0.8:", X.groupby("pure").apply(lambda d: (d.cultivated == d.map_cropped).mean()).round(3).to_dict())

# ---------- Comment 13
# B1 crop-class persistence between consecutive maps
W = CR.pivot(index="parcel_id", columns="map_year", values="crop")
pers = {}
for y in range(2019, 2024):
    a, b = W[y], W[y + 1]; ok = a.notna() & b.notna()
    pers[f"{y}->{y+1}"] = dict(n=int(ok.sum()), same_class=float((a[ok] == b[ok]).mean()),
                               same_excl_unclassified=float((a[ok & (a != "unclassified") & (b != "unclassified")] == b[ok & (a != "unclassified") & (b != "unclassified")]).mean()))
PERS = pd.DataFrame(pers).T; print(PERS.round(3)); PERS.to_csv(OUT / "TableS_R13a_cropclass_persistence.csv")
# persistence 2023->2024 by class (what a 2025 reuse implies)
a, b = W[2023], W[2024]; ok = a.notna() & b.notna()
byc = pd.crosstab(a[ok], b[ok], normalize="index").round(3); print(byc); byc.to_csv(OUT / "TableS_R13b_transition_2023_2024.csv")
stay = pd.Series({c: byc.loc[c, c] for c in byc.index}); print("stay-in-class 2023->2024:", stay.round(3).to_dict())
# transitions that change the crop calendar group: annual<->orchard
grp = lambda s: np.where(s == "orchard", "orchard", np.where(s == "unclassified", "uncl", "annual"))
print("2023->2024 annual->orchard:", float(((grp(a[ok]) == "annual") & (grp(b[ok]) == "orchard")).mean()), "orchard->annual:", float(((grp(a[ok]) == "orchard") & (grp(b[ok]) == "annual")).mean()))

# B2 forward validation with and without 2025
Em = E[(E.crop != "unclassified") & E.p_rec_36d.notna()]
te = Em[Em.year >= 2023]
def cidx(d): return concordance_index(d.t, 1 - d.p_rec_36d, d.recovered)
print("concordance test all 2023-25:", round(cidx(te), 4), "| 2023-24:", round(cidx(te[te.year <= 2024]), 4), "| 2025 only:", round(cidx(te[te.year == 2025]), 4))
for y in [2023, 2024, 2025]: print(" ", y, round(cidx(te[te.year == y]), 4), len(te[te.year == y]))
el = te[(te.t >= 36) | (te.recovered == 1)].copy(); el["y36"] = ((el.recovered == 1) & (el.t <= 36)).astype(int)
print("AUC36 2023-25:", round(roc_auc_score(el.y36, el.p_rec_36d), 4), "| 2023-24:", round(roc_auc_score(el[el.year <= 2024].y36, el[el.year <= 2024].p_rec_36d), 4), "| 2025:", round(roc_auc_score(el[el.year == 2025].y36, el[el.year == 2025].p_rec_36d), 4))

# B5 census with/without 2025
def km(d): return float(1 - KaplanMeierFitter().fit(d.t, d.recovered).survival_function_at_times(36).values[0])
for lab, d in [("all", E), ("excl 2025", E[E.year <= 2024]), ("2025", E[E.year == 2025])]:
    print(lab, "episodes", len(d), "unresolved", round((d.recovered == 0).mean(), 3), "P36", round(km(d), 3))

# B3 exit with and without 2025
by = [2019, 2020, 2021, 2022]
pyb = PY[PY.year.isin(by)].groupby("parcel_id").cultivated.sum().rename("cult_base")
Eb = E[E.year.isin(by)].groupby("parcel_id").agg(eps_base=("dur_d", "size"), nonrec_share=("recovered", lambda s: 1 - s.mean()), nonrec_days_base=("nonrec_days", "sum"))
S = pd.read_parquet(UP / "interim/parcels_struct_full.parquet").drop(columns="geometry")
hyd = pd.read_csv(ROOT / "hydraulic" / "hydraulic_position.csv")
S = S.drop(columns=[c for c in hyd.columns if c != "parcel_id" and c in S.columns]).merge(hyd, on="parcel_id", how="left")
S["hyd_tier"] = S.hyd_tier.fillna(3).astype(int); S["hyd_unmapped"] = (S.hyd_tier != 1).astype(int)
S["hyd_rank"] = S.hyd_rank_cmd.fillna(S.hyd_rank_county); S["hyd_rank"] = S.hyd_rank.fillna(S.groupby("county").hyd_rank.transform("median")).fillna(0.5)
S["hyd_dz_m"] = S.hyd_dz_m.fillna(S.groupby("county").hyd_dz_m.transform("median")).fillna(0)
F3 = "exit ~ deficit_days + exposure + nonrec_share + log_area + log_dist_desert + hyd_rank + hyd_dz_m + hyd_unmapped + C(salinity_class, Treatment('mid')) + log_ntl + dist_settlement_km + dist_road_km + C(county)"
for lab, ty in [("2023-25", [2023, 2024, 2025]), ("2023-24", [2023, 2024])]:
    pyt = PY[PY.year.isin(ty)].groupby("parcel_id").cultivated.agg(cult_test="sum", n_test="size")
    X3 = S.merge(pyb, on="parcel_id").merge(pyt, on="parcel_id").merge(Eb, on="parcel_id", how="left")
    X3 = X3[X3.cult_base >= 3].copy()
    X3["exposure"] = X3.eps_base.fillna(0) / X3.cult_base; X3["deficit_days"] = X3.nonrec_days_base.fillna(0) / X3.cult_base; X3["nonrec_share"] = X3.nonrec_share.fillna(0)
    X3["exit"] = ((X3.n_test - X3.cult_test) >= 1).astype(int); X3["cell_code"] = pd.factorize(X3.cell)[0]
    m = smf.logit(F3, data=X3).fit(disp=0, cov_type="cluster", cov_kwds={"groups": X3.cell_code.values}, maxiter=300)
    print(f"exit {lab}: n={len(X3)}, exits={X3.exit.sum()} ({X3.exit.mean():.2%}), OR deficit={np.exp(m.params['deficit_days']):.4f} [{np.exp(m.conf_int().loc['deficit_days',0]):.4f}-{np.exp(m.conf_int().loc['deficit_days',1]):.4f}] p={m.pvalues['deficit_days']:.1e}; OR exposure={np.exp(m.params['exposure']):.3f}")

# B4 targeting with/without 2025
base = E[E.year.isin(by)].groupby("parcel_id").agg(eps=("dur_d", "size"), nonrec=("nonrec_days", "sum")).reindex(S.parcel_id).fillna(0)
base = base.join(pyb).dropna(); base["exposure"] = base.eps / base.cult_base; base["deficit"] = base.nonrec / base.cult_base
rng = np.random.default_rng(20260913)
for lab, ty in [("2023-25", [2023, 2024, 2025]), ("2023-24", [2023, 2024])]:
    tgt = E[E.year.isin(ty)].groupby("parcel_id").nonrec_days.sum().reindex(base.index).fillna(0)
    k = int(0.2 * len(base)); tot = tgt.sum()
    res = {}
    for name, order in [("exposure", base.exposure), ("deficit", base.deficit)]:
        top = order.sort_values(ascending=False).index[:k]; res[name] = tgt.loc[top].sum() / tot
    res["random"] = np.mean([tgt.loc[rng.choice(base.index, k, replace=False)].sum() / tot for _ in range(20)])
    print(f"targeting {lab} (20% budget, n={len(base)}):", {k_: round(v, 3) for k_, v in res.items()})


## 3.2 · Cultivation threshold sensitivity (EVI2max ≥ 0.20 / 0.25 / 0.30)

In [ ]:
import os, numpy as np, pandas as pd, statsmodels.formula.api as smf
from pathlib import Path
ROOT = Path(os.environ.get("TIPTAP_ROOT", r"D:\BRA\TipTap")); UP = ROOT / "outputs_recovery"; OUT = UP / "revision" / "R12_R13"
H=pd.read_csv(UP/"revision/R11_parcel_heterogeneity.csv"); print(H.shape, H.columns.tolist())
PS=pd.read_csv(UP/"gis/parcel_season_nonrec.csv",usecols=["parcel_id","year"]).assign(cult025=1)
H=H.merge(PS,on=["parcel_id","year"],how="left").fillna({"cult025":0})
print("agreement of evi2_max_all>=0.25 with pipeline flag:", ((H.evi2_max_all>=0.25).astype(int)==H.cult025).mean())
CR=pd.read_parquet(OUT/"crop_majority_2019_2024.parquet").rename(columns={"map_year":"year"})
X=H.merge(CR[["parcel_id","year","crop"]],on=["parcel_id","year"],how="left")
X["map_cropped"]=(X.crop.notna()&(X.crop!="unclassified")).astype(int)
E=pd.read_parquet(UP/"interim/episodes_full_final.parquet")
S=pd.read_parquet(UP/"interim/parcels_struct_full.parquet").drop(columns="geometry")
hyd=pd.read_csv(ROOT / "hydraulic" / "hydraulic_position.csv")
S=S.drop(columns=[c for c in hyd.columns if c!="parcel_id" and c in S.columns]).merge(hyd,on="parcel_id",how="left")
S["hyd_tier"]=S.hyd_tier.fillna(3).astype(int); S["hyd_unmapped"]=(S.hyd_tier!=1).astype(int)
S["hyd_rank"]=S.hyd_rank_cmd.fillna(S.hyd_rank_county); S["hyd_rank"]=S.hyd_rank.fillna(S.groupby("county").hyd_rank.transform("median")).fillna(0.5)
S["hyd_dz_m"]=S.hyd_dz_m.fillna(S.groupby("county").hyd_dz_m.transform("median")).fillna(0)
by=[2019,2020,2021,2022]; ty=[2023,2024,2025]
Eb=E[E.year.isin(by)].groupby("parcel_id").agg(eps_base=("dur_d","size"),nonrec_share=("recovered",lambda s:1-s.mean()),nonrec_days_base=("nonrec_days","sum"))
F3="exit ~ deficit_days + exposure + nonrec_share + log_area + log_dist_desert + hyd_rank + hyd_dz_m + hyd_unmapped + C(salinity_class, Treatment('mid')) + log_ntl + dist_settlement_km + dist_road_km + C(county)"
rows=[]
for thr in [0.20,0.25,0.30]:
    H["cult"]=(H.evi2_max_all>=thr).astype(int); Xm=X.assign(cult=(X.evi2_max_all>=thr).astype(int)); Xm=Xm[Xm.crop.notna()&(Xm.year<=2024)]
    tp=((Xm.cult==1)&(Xm.map_cropped==1)).sum(); fn=((Xm.cult==0)&(Xm.map_cropped==1)).sum(); tn=((Xm.cult==0)&(Xm.map_cropped==0)).sum(); fp=((Xm.cult==1)&(Xm.map_cropped==0)).sum()
    sens=tp/(tp+fn); spec=tn/(tn+fp)
    pyb=H[H.year.isin(by)].groupby("parcel_id").cult.sum().rename("cult_base"); pyt=H[H.year.isin(ty)].groupby("parcel_id").cult.agg(cult_test="sum",n_test="size")
    X3=S.merge(pyb,on="parcel_id").merge(pyt,on="parcel_id").merge(Eb,on="parcel_id",how="left"); X3=X3[X3.cult_base>=3].copy()
    X3["exposure"]=X3.eps_base.fillna(0)/X3.cult_base; X3["deficit_days"]=X3.nonrec_days_base.fillna(0)/X3.cult_base; X3["nonrec_share"]=X3.nonrec_share.fillna(0)
    X3["exit"]=((X3.n_test-X3.cult_test)>=1).astype(int); X3["cell_code"]=pd.factorize(X3.cell)[0]
    m=smf.logit(F3,data=X3).fit(disp=0,cov_type="cluster",cov_kwds={"groups":X3.cell_code.values},maxiter=300)
    ci=np.exp(m.conf_int().loc["deficit_days"])
    rows.append(dict(threshold=thr,cultivated_share=H.cult.mean(),sensitivity_vs_map=sens,specificity_vs_map=spec,balanced_acc=0.5*(sens+spec),
                     fallow_share_map_unclassified=1-Xm[Xm.cult==0].map_cropped.mean(),cultivated_share_map_unclassified=1-Xm[Xm.cult==1].map_cropped.mean(),
                     exit_n=len(X3),exit_events=int(X3.exit.sum()),exit_rate=X3.exit.mean(),OR_deficit=np.exp(m.params["deficit_days"]),OR_lo=ci[0],OR_hi=ci[1],p=m.pvalues["deficit_days"]))
    print(rows[-1])
T=pd.DataFrame(rows); T.to_csv(OUT/"TableS_R12c_threshold_sensitivity.csv",index=False); print(T.round(4).to_string())


## 4.1 · Setup and inputs

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import proportional_hazard_test
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"savefig.dpi": 600, "font.size": 8, "axes.spines.top": False, "axes.spines.right": False})

ROOT = Path(os.environ.get("TIPTAP_ROOT", r"D:\BRA\TipTap"))
OUT = Path(os.environ.get("TIPTAP_RCHECK_OUT", str(ROOT / "outputs_recovery" / "revision" / "R_checks")))
OUT.mkdir(parents=True, exist_ok=True)
BASE, TEST = [2019, 2020, 2021, 2022], [2023, 2024, 2025]
SEED = 20260913

In [ ]:
E = pd.read_parquet(ROOT / "outputs_recovery/interim/episodes_full_final.parquet")
S = pd.read_parquet(ROOT / "outputs_recovery/interim/parcels_struct_full.parquet").drop(columns="geometry")
hyd = pd.read_csv(ROOT / "hydraulic/hydraulic_position.csv")
S = S.drop(columns=[c for c in hyd.columns if c != "parcel_id" and c in S.columns]).merge(hyd, on="parcel_id", how="left")
S["hyd_tier"] = S.hyd_tier.fillna(3).astype(int); S["hyd_unmapped"] = (S.hyd_tier != 1).astype(int)
S["hyd_rank"] = S.hyd_rank_cmd.fillna(S.hyd_rank_county)
S["hyd_rank"] = S.hyd_rank.fillna(S.groupby("county").hyd_rank.transform("median")).fillna(0.5)
S["hyd_dz_m"] = S.hyd_dz_m.fillna(S.groupby("county").hyd_dz_m.transform("median")).fillna(0)

PS = pd.read_csv(ROOT / "outputs_recovery/gis/parcel_season_nonrec.csv", usecols=["parcel_id", "year"]).assign(cultivated=1)
cov = {y: pd.read_parquet(ROOT / f"outputs_recovery/interim/parcel_meta_full_{y}.parquet", columns=["parcel_id"]).parcel_id for y in range(2019, 2026)}
PY = pd.concat([pd.DataFrame({"parcel_id": v, "year": y}) for y, v in cov.items()]).merge(PS, how="left").fillna({"cultivated": 0})
PY["cultivated"] = PY.cultivated.astype(int)
crop_file = next(p for p in [ROOT / "outputs_recovery/revision/R12_R13/crop_majority_2019_2024.parquet", OUT.parent / "R12_R13/crop_majority_2019_2024.parquet"] if p.exists())
CROP = pd.read_parquet(crop_file).rename(columns={"map_year": "year"})

assert len(E) == 266_328 and len(S) == 127_245
print(f"episodes {len(E):,} | parcels {len(S):,} | parcel-seasons {len(PY):,} | cultivated {PY.cultivated.sum():,}")

## 4.2 · Survival model: dependence, proportional hazards, penalty and full follow-up

Specification as in the manuscript (1.11). Variants: (a) as published, cluster = 9-km cell; (b) cluster = parcel; (c) no penalty; (d) stratified by onset stage; (e) acute onsets with at least 36 d to harvest.

In [ ]:
NUM = ["log_area", "hyd_rank", "hyd_dz_m", "hyd_unmapped", "log_dist_desert", "mulch", "log_ntl", "dist_settlement_km", "dist_road_km", "z_onset", "demand"]
CAT = {"crop": "cotton", "onset_stage": "development", "mechanism": "water-led", "salinity_class": "mid", "year": 2021}
STRUCT = ["log_area", "hyd_rank", "hyd_dz_m", "hyd_unmapped", "log_dist_desert", "mulch", "log_ntl", "dist_settlement_km", "dist_road_km", "salinity_class_high", "salinity_class_low"]

def design(df, drop_cat=()):
    X = df[NUM].copy()
    for c, r in CAT.items():
        if c in drop_cat: continue
        X = pd.concat([X, pd.get_dummies(df[c].astype(str), prefix=c, dtype=float).drop(columns=[f"{c}_{r}"], errors="ignore")], axis=1)
    return X.fillna(0)

def fit_cox(e, cluster="cell", penalizer=0.01, strata=None):
    e = e[e.crop != "unclassified"].copy()
    e["cluster_id"] = pd.factorize(e[cluster])[0]
    X = design(e, drop_cat=(strata,) if strata else ())
    tr = e.year.isin(BASE).values
    Xtr = X[tr].copy(); Xtr["t"] = e.t[tr].values; Xtr["recovered"] = e.recovered[tr].values; Xtr["cluster_id"] = e.cluster_id[tr].values
    if strata: Xtr[strata] = e[strata][tr].values
    Xtr = Xtr.loc[:, (Xtr.std(numeric_only=True) > 0).reindex(Xtr.columns, fill_value=True)]
    cph = CoxPHFitter(penalizer=penalizer, strata=[strata] if strata else None)
    cph.fit(Xtr, duration_col="t", event_col="recovered", cluster_col="cluster_id", robust=True)
    feats = [c for c in Xtr.columns if c not in ("t", "recovered", "cluster_id", strata)]
    Xte = X[~tr][feats].copy()
    if strata: Xte[strata] = e[strata][~tr].values
    c_tr = concordance_index(e.t[tr], -cph.predict_partial_hazard(Xtr[feats + ([strata] if strata else [])]), e.recovered[tr])
    c_te = concordance_index(e.t[~tr], -cph.predict_partial_hazard(Xte), e.recovered[~tr])
    hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].set_axis(["HR", "lo", "hi", "p"], axis=1)
    return cph, hr, dict(n_train=int(tr.sum()), n_test=int((~tr).sum()), c_train=c_tr, c_test=c_te), Xtr

FITS, HR, INFO, XTR = {}, {}, {}, {}
E["full_followup"] = (E.acute == 1) & ((E.end_cidx - E.onset_cidx) * 12 >= 36) | ((E.acute == 1) & (E.recovered == 1))
variants = {
    "published (cell-clustered, ridge 0.01)": dict(e=E),
    "parcel-clustered": dict(e=E, cluster="parcel_id"),
    "unpenalised": dict(e=E, penalizer=0.0),
    "stratified by onset stage": dict(e=E, strata="onset_stage"),
    "acute onsets with >= 36 d follow-up": dict(e=E[E.full_followup]),
}
for name, kw in variants.items():
    t0 = time.time(); FITS[name], HR[name], INFO[name], XTR[name] = fit_cox(**kw)
    print(f"{name}: {INFO[name]} ({time.time()-t0:.0f}s)", flush=True)

pub = pd.read_csv(ROOT / "outputs_recovery/tables/table2_cox_hazard_ratios.csv", index_col=0)
assert np.allclose(pub.HR, HR["published (cell-clustered, ridge 0.01)"].HR.reindex(pub.index)), "published fit not reproduced"
cmp = pd.concat({k: v.reindex(STRUCT) for k, v in HR.items()}, axis=1)
cmp.to_csv(OUT / "TableS_cox_variants.csv"); pd.DataFrame(INFO).T.to_csv(OUT / "TableS_cox_variants_fit.csv")
cmp.xs("HR", axis=1, level=1).round(3)

In [ ]:
cph0, Xtr0 = FITS["published (cell-clustered, ridge 0.01)"], XTR["published (cell-clustered, ridge 0.01)"]
e0 = E[E.crop != "unclassified"]
Xtr0 = Xtr0.assign(cluster_id=pd.factorize(e0.cell)[0][e0.year.isin(BASE).values])
ph = proportional_hazard_test(cph0, Xtr0, time_transform="rank")
PH = ph.summary.sort_values("p"); PH.to_csv(OUT / "TableS_schoenfeld.csv")
print("covariates violating PH at p < 0.01:", int((PH.p < 0.01).sum()), "of", len(PH))
PH.head(12).round(4)

## 4.3 · Forward calibration and recalibration (Fig. 12)

Calibration of the published 36-day recovery probability on 2023–2024 (observed crop type), individual-level logistic recalibration (intercept and slope on the logit), and decile tables before and after.

In [ ]:
el = E[(E.crop != "unclassified") & E.p_rec_36d.notna() & E.year.isin(TEST) & ((E.t >= 36) | (E.recovered == 1))].copy()
el["y36"] = ((el.recovered == 1) & (el.t <= 36)).astype(int)
el["logit_p"] = np.log(el.p_rec_36d / (1 - el.p_rec_36d))
rows = {}
for lab, d in [("2023-2025", el), ("2023-2024", el[el.year <= 2024]), ("2025", el[el.year == 2025])]:
    m = sm.Logit(d.y36, sm.add_constant(d.logit_p)).fit(disp=0)
    d = d.copy(); d["p_recal"] = m.predict(sm.add_constant(d.logit_p))
    rows[lab] = dict(n=len(d), observed=d.y36.mean(), predicted=d.p_rec_36d.mean(), cal_intercept=m.params["const"], cal_slope=m.params["logit_p"],
                     auc=roc_auc_score(d.y36, d.p_rec_36d), brier_before=((d.p_rec_36d - d.y36) ** 2).mean(), brier_after=((d.p_recal - d.y36) ** 2).mean())
CAL = pd.DataFrame(rows).T; CAL.to_csv(OUT / "TableS_calibration.csv"); CAL.round(4)

In [ ]:
d = el[el.year <= 2024].copy()
m = sm.Logit(d.y36, sm.add_constant(d.logit_p)).fit(disp=0); d["p_recal"] = m.predict(sm.add_constant(d.logit_p))
d["decile"] = pd.qcut(d.p_rec_36d.rank(method="first"), 10, labels=False) + 1
DEC = d.groupby("decile").agg(n=("y36", "size"), observed=("y36", "mean"), predicted=("p_rec_36d", "mean"), recalibrated=("p_recal", "mean"))
DEC.to_csv(OUT / "TableS_calibration_deciles_2023_2024.csv")
fig, ax = plt.subplots(figsize=(3.4, 3.2))
ax.plot(DEC.predicted, DEC.observed, "o-", ms=3, color="#eb0de3", label="as published")
ax.plot(DEC.recalibrated, DEC.observed, "s-", ms=3, color="#0de7e0", label="recalibrated")
lim = (0, max(DEC.observed.max(), DEC.predicted.max()) * 1.05); ax.plot(lim, lim, "k--", lw=.6); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("predicted P(recover ≤ 36 d)"); ax.set_ylabel("observed"); ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "FigS_calibration_2023_2024.pdf", bbox_inches="tight"); plt.show()
DEC.round(3)

## 4.4 · Exit outcome: stricter definitions and absolute risk

In [ ]:
pyb = PY[PY.year.isin(BASE)].groupby("parcel_id").cultivated.sum().rename("cult_base")
W = PY[PY.year.isin(TEST)].pivot(index="parcel_id", columns="year", values="cultivated")
X3 = S.merge(pyb, on="parcel_id").merge(W, on="parcel_id")
X3 = X3[X3.cult_base >= 3].copy()
X3["exit1"] = (X3[TEST].eq(0).sum(axis=1) >= 1).astype(int)
X3["exit2"] = (((X3[2023] == 0) & (X3[2024] == 0)) | ((X3[2024] == 0) & (X3[2025] == 0))).astype(int)
cm = CROP.pivot(index="parcel_id", columns="year", values="crop")
X3["exit_map"] = X3.parcel_id.map(((cm[2023] == "unclassified") & (cm[2024] == "unclassified")).astype(int)).fillna(0).astype(int)
Eb = E[E.year.isin(BASE)].groupby("parcel_id").agg(eps_base=("dur_d", "size"), nonrec_share=("recovered", lambda s: 1 - s.mean()), nonrec_days_base=("nonrec_days", "sum"))
X3 = X3.merge(Eb, on="parcel_id", how="left")
X3["exposure"] = X3.eps_base.fillna(0) / X3.cult_base; X3["deficit_days"] = X3.nonrec_days_base.fillna(0) / X3.cult_base; X3["nonrec_share"] = X3.nonrec_share.fillna(0)
X3["cell_code"] = pd.factorize(X3.cell)[0]
F3 = "{y} ~ deficit_days + exposure + nonrec_share + log_area + log_dist_desert + hyd_rank + hyd_dz_m + hyd_unmapped + C(salinity_class, Treatment('mid')) + log_ntl + dist_settlement_km + dist_road_km + C(county)"
rows = []
for y in ["exit1", "exit2", "exit_map"]:
    m = smf.logit(F3.format(y=y), data=X3).fit(disp=0, cov_type="cluster", cov_kwds={"groups": X3.cell_code.values}, maxiter=300)
    ame = m.get_margeff(at="overall").summary_frame().loc["deficit_days"]
    pred = {k: m.predict(X3.assign(deficit_days=k)).mean() for k in (0, 30, 60)}
    ci = np.exp(m.conf_int().loc["deficit_days"])
    rows.append(dict(outcome=y, n=len(X3), events=int(X3[y].sum()), prevalence=X3[y].mean(), OR_per_day=np.exp(m.params["deficit_days"]), OR_lo=ci[0], OR_hi=ci[1],
                     AME_pp_per_day=100 * ame["dy/dx"], risk_at_0=pred[0], risk_at_30=pred[30], risk_at_60=pred[60]))
EXIT = pd.DataFrame(rows).set_index("outcome"); EXIT.to_csv(OUT / "TableS_exit_definitions.csv"); EXIT.round(4)

## 4.5 · Recovery Capacity Index: dependence on the reference episode

RCI = Cox-predicted P(recover ≤ 36 d) for a parcel's structural covariates under a fixed reference episode. Three references; Spearman correlation of the resulting parcel rankings.

In [ ]:
ref = S[["parcel_id", "log_area", "hyd_rank", "hyd_dz_m", "hyd_unmapped", "log_dist_desert", "log_ntl", "dist_settlement_km", "dist_road_km", "salinity_class"]].copy()
ref["mulch"] = ref.parcel_id.map(E.groupby("parcel_id").mulch.mean().round()).fillna(0)
trn = E[E.year.isin(BASE)]
ref["z_onset"] = float(trn.z_onset.median()); ref["demand"] = float(trn.demand.median())
REFS = {"published: cotton, development, water-led, 2021": dict(crop="cotton", onset_stage="development", mechanism="water-led", year=2021),
        "orchard, mid-season, chronic, 2022": dict(crop="orchard", onset_stage="mid-season", mechanism="chronic", year=2022),
        "cotton, initial, idiosyncratic, 2019": dict(crop="cotton", onset_stage="initial", mechanism="idiosyncratic", year=2019)}
feats = [c for c in Xtr0.columns if c not in ("t", "recovered", "cluster_id")]
RCI = pd.DataFrame({"parcel_id": ref.parcel_id})
for name, r in REFS.items():
    X = design(ref.assign(**r)).reindex(columns=feats, fill_value=0)
    RCI[name] = 1 - cph0.predict_survival_function(X, times=[36]).loc[36].values
rho = pd.DataFrame({a: {b: spearmanr(RCI[a], RCI[b])[0] for b in REFS} for a in REFS})
RCI.describe().T[["mean", "25%", "50%", "75%"]].round(3).to_csv(OUT / "TableS_rci_reference_levels.csv"); rho.to_csv(OUT / "TableS_rci_reference_spearman.csv")
print(RCI.describe().T[["mean", "50%"]].round(3)); rho.round(3)

## 4.6 · MGWR diagnostics

From the ArcGIS Pro outputs: bandwidths per covariate, global fit of the local model, condition numbers, and residual spatial autocorrelation (Moran's I, 6 nearest hexagon neighbours).

In [ ]:
bw = pd.read_csv(ROOT / "hydraulic/spatial/mgwr_bandwidths.csv")[["VAR_ALIAS", "NBR_COUNT", "NBR_PCT", "SIG_PCT", "C_MEAN", "C_MEDIAN", "C_MIN", "C_MAX"]]
hx = pd.read_csv(ROOT / "hydraulic/spatial/hex_mgwr.csv")
import geopandas as gpd
grid = gpd.read_file(ROOT / "hydraulic/spatial/hex_grid.gpkg")
key = "GRID_ID" if "GRID_ID" in grid.columns else grid.columns[0]
hx = hx.merge(grid[[key, "geometry"]], left_on="GRID_ID", right_on=key, how="left"); hx = gpd.GeoDataFrame(hx, geometry="geometry", crs=grid.crs)
xy = np.c_[hx.geometry.centroid.x, hx.geometry.centroid.y]
from scipy.spatial import cKDTree
_, nb_idx = cKDTree(xy).query(xy, k=7); nb_idx = nb_idx[:, 1:]
def moran(z):
    z = (z - z.mean()); lag = z.values[nb_idx].mean(axis=1)
    return float((z * lag).sum() / (z ** 2).sum())
y, yhat, res = hx.S_LOG_DEFICIT, hx.S_PREDICT, hx.S_RESIDUAL
diag = dict(hexagons=len(hx), r2_local_model=1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            condition_number_median=float(hx.CND_NUMBER.median()), condition_number_max=float(hx.CND_NUMBER.max()), share_condition_gt_30=float((hx.CND_NUMBER > 30).mean()),
            moran_I_residual=moran(res), moran_I_dependent=moran(y))
pd.Series(diag).to_csv(OUT / "TableS_mgwr_diagnostics.csv"); bw.to_csv(OUT / "TableS_mgwr_bandwidths.csv", index=False)
print(pd.Series(diag).round(4)); bw.round(3)

## 4.7 · Pre-season salinity from baseline-year imagery only

The first cell writes `revision/R34_si_preseason_base.csv` (median March SI, 2019–2022); salinity terciles are rebuilt within county and the Cox and exit models refitted.

In [ ]:
import pyarrow.parquet as pq
rec = ROOT / "outputs_recovery"
si = pd.concat([pq.read_table(rec / "interim" / f"panel_full_{y}.parquet", columns=["parcel_id", "cidx", "SI_mean"]).to_pandas().query("cidx <= 2") for y in [2019, 2020, 2021, 2022]])
si.groupby("parcel_id").SI_mean.median().rename("si_preseason").reset_index().to_csv(rec / "revision" / "R34_si_preseason_base.csv", index=False)

In [ ]:
f = ROOT / "outputs_recovery/revision/R34_si_preseason_base.csv"
if f.exists():
    si = pd.read_csv(f)
    S2 = S.drop(columns=["si_preseason", "salinity_class"]).merge(si, on="parcel_id", how="left")
    S2["salinity_class"] = S2.groupby("county").si_preseason.transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["low", "mid", "high"]))
    same = (S2.set_index("parcel_id").salinity_class.reindex(S.parcel_id).values == S.salinity_class.values).mean()
    E2 = E.drop(columns=["si_preseason", "salinity_class"]).merge(S2[["parcel_id", "si_preseason", "salinity_class"]], on="parcel_id", how="left")
    _, hr2, info2, _ = fit_cox(E2)
    X3b = X3.drop(columns=["salinity_class", "si_preseason"]).merge(S2[["parcel_id", "salinity_class", "si_preseason"]], on="parcel_id")
    m2 = smf.logit(F3.format(y="exit1"), data=X3b).fit(disp=0, cov_type="cluster", cov_kwds={"groups": X3b.cell_code.values}, maxiter=300)
    SAL = pd.DataFrame({"published (all years)": HR["published (cell-clustered, ridge 0.01)"].HR.reindex(STRUCT), "baseline years only": hr2.HR.reindex(STRUCT)})
    SAL.loc["exit OR per deficit-day"] = [EXIT.loc["exit1", "OR_per_day"], np.exp(m2.params["deficit_days"])]
    SAL.loc["exit OR high salinity"] = [np.nan, np.exp(m2.params["C(salinity_class, Treatment('mid'))[T.high]"])]
    SAL.to_csv(OUT / "TableS_salinity_baseline_only.csv")
    print(f"salinity class unchanged for {same:.1%} of parcels; forward concordance {info2['c_test']:.3f}"); display(SAL.round(3))
else:
    print("R34_si_preseason_base.csv not found — run appendix cell A3 in the pipeline notebook, then re-run this cell")

In [ ]:
tab = pd.DataFrame({"OR": np.exp(m2.params), "lo": np.exp(m2.conf_int()[0]), "hi": np.exp(m2.conf_int()[1]), "p": m2.pvalues})
tab = tab.loc[[i for i in tab.index if not i.startswith(("C(county", "Intercept"))]]
tab.to_csv(OUT / "TableS_exit_salinity_baseline_full.csv"); tab.round(3)

## 4.8 · Table 4: exit from cultivation and annual-to-orchard conversion

In [ ]:
# Table 4, annual-to-orchard transition: annual crop in every 2019–2022 map and orchard in both the 2023 and 2024 maps,
# modelled on parcels with an annual crop throughout the baseline (all-season and baseline-only salinity)
cm_all = CROP.pivot(index="parcel_id", columns="year", values="crop")
annual = cm_all[BASE].apply(lambda s: s.notna() & ~s.isin(["orchard", "unclassified"])).all(axis=1)
conv = (cm_all[2023] == "orchard") & (cm_all[2024] == "orchard")
ORCH = {}
for lab, d in [("all-season salinity", X3), ("baseline-only salinity", X3b)]:
    d = d[d.parcel_id.map(annual).fillna(False).astype(bool)].copy()
    d["orchard_conv"] = d.parcel_id.map(conv).fillna(False).astype(int)
    m = smf.logit(F3.format(y="orchard_conv"), data=d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": d.cell_code.values}, maxiter=300)
    t = pd.DataFrame({"OR": np.exp(m.params), "lo": np.exp(m.conf_int()[0]), "hi": np.exp(m.conf_int()[1]), "p": m.pvalues})
    ORCH[lab] = t.loc[[i for i in t.index if not i.startswith(("C(county", "Intercept"))]]
    print(f"{lab}: n={len(d):,}, events={int(d.orchard_conv.sum()):,}")
pd.concat(ORCH, axis=1).to_csv(OUT / "TableS_orchard_strict_salinity.csv")
TABLE4 = pd.concat({"exit": tab, "orchard_conv": ORCH["baseline-only salinity"]}, axis=1)
TABLE4.to_csv(OUT / "Table4_landuse_change.csv"); TABLE4.round(3)

## 4.9 · Targeting uncertainty and tier-1 head-to-tail refit

In [ ]:
# 9 · Targeting uncertainty and tier-1 head-to-tail refit
SEED = 20260913
rng = np.random.default_rng(SEED)

pyb = PY[PY.year.isin(BASE)].groupby("parcel_id").cultivated.sum().rename("cult_base")
base = (E[E.year.isin(BASE)].groupby("parcel_id").agg(eps=("dur_d", "size"), nonrec=("nonrec_days", "sum"))
          .reindex(S.parcel_id).fillna(0).join(pyb, how="inner").join(S.set_index("parcel_id").cell))
base = base[base.cult_base > 0]
base["exposure"] = base.eps / base.cult_base
base["deficit"] = base.nonrec / base.cult_base
tgt = E[E.year.isin(TEST)].groupby("parcel_id").nonrec_days.sum().reindex(base.index).fillna(0).values
score = {r: base[r].values for r in ("exposure", "deficit")}
groups = pd.Series(np.arange(len(base))).groupby(pd.factorize(base.cell)[0]).apply(np.array).values

def capture(idx, budget):
    k = int(budget * len(idx)); t = tgt[idx]; tot = max(t.sum(), 1e-9); u = rng.random(len(idx))
    return [t[np.lexsort((u, -score[r][idx]))[:k]].sum() / tot for r in ("exposure", "deficit")]

rows = []
for budget in (0.2, 0.3):
    pt = np.mean([capture(np.arange(len(base)), budget) for _ in range(100)], axis=0)
    bs = np.array([capture(np.concatenate(groups[rng.integers(0, len(groups), len(groups))]), budget) for _ in range(500)])
    d = bs[:, 1] - bs[:, 0]
    rows.append(dict(budget=budget, n_parcels=len(base), n_cells=len(groups),
                     exposure=pt[0], exposure_lo=np.percentile(bs[:, 0], 2.5), exposure_hi=np.percentile(bs[:, 0], 97.5),
                     deficit=pt[1], deficit_lo=np.percentile(bs[:, 1], 2.5), deficit_hi=np.percentile(bs[:, 1], 97.5),
                     diff=pt[1] - pt[0], diff_lo=np.percentile(d, 2.5), diff_hi=np.percentile(d, 97.5),
                     p_deficit_better=(d > 0).mean()))
TGT = pd.DataFrame(rows)
TGT.to_csv(OUT / "TableS_targeting_bootstrap.csv", index=False)
display(TGT.round(3))

_, hr_t1, info_t1, _ = fit_cox(E[E.hyd_tier == 1])
T1 = pd.DataFrame({"pooled": HR["published (cell-clustered, ridge 0.01)"].HR, "tier-1 only": hr_t1.HR}).reindex(STRUCT)
T1.to_csv(OUT / "TableS_tier1_headtail.csv")
print(info_t1)
print("tier-1 episodes:", (E.hyd_tier == 1).sum(), f"({(E.hyd_tier == 1).mean():.1%} of all)")
print("hyd_rank:", hr_t1.loc["hyd_rank"].round(3).to_dict())